<a href="https://colab.research.google.com/github/CyberWarSmith/AXIOM/blob/main/SL_v0_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SL v0.1 pilot notebook

This file is the single versioned source of truth for every cell that is
run in Colab. Edit here, not on the Notion page. Rebuilt 2026-08-05 from
the Notion page body, which was ahead of the previous .ipynb on 14 cells.

**Step versus Cell.** `Step NN` is the physical position in this notebook
and is the order you run in, top to bottom. `Cell N` is the stable
identity used by every cross reference on the Notion page, in the handover
and in the pending task list. Cells are deliberately NOT renumbered: the
suffix letters (3b, 5b, 10b, 12b, 16b) mark cells added after the original
numbering was fixed. If the two ever disagree, `Cell N` is the identity
and `Step NN` is only a position.

**Changed in this rebuild.** Cell 10b moved from before Cell 11 to after
Cell 12, which is the only genuine ordering fault: its first block calls
`score_mechanical` (Cell 11) and its second calls `judge_trial` (Cell 12)
at execution time. The `schema_for` docstring in Cell 6 was corrected: it
claimed Cell 7 must run first, which is false, because the Cell 7 names
resolve at call time and the first call is in Cell 10.


# Step 00 · Cell 0: cleanup


In [ ]:
!rm -rf /content/SL_v0_1_PILOT

# Step 01 · Cell 1: config


In [ ]:
!pip -q install openai tenacity numpy pandas scipy tqdm

import os, json, hashlib, random, itertools, time, textwrap, re
from pathlib import Path
import numpy as np, pandas as pd
from google.colab import userdata

SEED = 20260804
random.seed(SEED); np.random.seed(SEED)

HF_TOKEN_SECRET = 'HuggingFaceNew'
API_KEY = userdata.get(HF_TOKEN_SECRET)

# CORRECTED 2026-08-04. The secret is a HuggingFace token, so it must go to
# the HF Inference Providers router, not to DeepInfra directly. A HF token
# against api.deepinfra.com returns 401 invalid_api_key. DeepInfra is reached
# as a routed provider via the ':deepinfra' model suffix in Cell 2.
BASE_URL = 'https://router.huggingface.co/v1'

# If you would rather call DeepInfra directly, create a DeepInfra API key,
# store it as a separate Colab secret, and set:
#   BASE_URL = 'https://api.deepinfra.com/v1/openai'
#   API_KEY  = userdata.get('DeepInfraKey')
# Do not mix the two. The token and the base URL must match.

RUN_ID   = 'SL_v0_1'
PILOT    = True          # Cell 8 enforces the reduced grid while True
# Pilot output is isolated from confirmatory output. Without the suffix both
# write to the same gen.jsonl, and generate_all's resume logic would silently
# adopt pilot trials as confirmatory data. Pilot trials are run before the
# hash exists, so that would contaminate the pre-registration at the source.
OUT      = Path(f'/content/{RUN_ID}' + ('_PILOT' if PILOT else ''))
OUT.mkdir(exist_ok=True)
CKPT     = OUT / 'checkpoints'; CKPT.mkdir(exist_ok=True)

RUNS_PER_CELL   = 3
MAX_LOOP_ITERS  = 3       # tier 0 -> 1 -> 2, then forced terminal
# Token budgeting removed 2026-08-04 after pilot 2. The per-trial budget was
# divided across the remaining iterations for loop arms only, so arm E's first
# call was capped at 1000 tokens while arms A, B, F and G each received the
# full 3000. The cap was tightest on exactly the arms under test, and arm E was
# the only arm that truncated. One symmetric ceiling replaces it: the same
# limit for every arm, every model and every iteration, far above the measured
# p95 of 518. Cost is unaffected, because output tokens are billed as produced
# rather than as permitted.
MAX_TOKENS_PER_CALL = 4096
TAU             = 0.60    # pre-registered point threshold
BOOTSTRAP_N     = 5000

# Escape safety, added 2026-08-04 after the Cell 10 JSONDecodeError.
# Several cells used a backslash-n escape inside ordinary quoted strings and
# the escape did not survive editing, so the code emitted a two character
# sequence where a newline was intended. NL and BS are built by codepoint, so
# no transcription or copy step can corrupt them.
NL = chr(10)
BS = chr(92)

GATES = {
    'blinding_max_recall'   : 0.45,
    'irr_min_kappa'         : 0.40,
    'fabricated_field_max'  : 0.05,
    'negative_control_max'  : 1.0,
}

print('config loaded', RUN_ID, 'PILOT' if PILOT else 'FULL')

config loaded SL_v0_1 PILOT


# Cell 1b: data staging and integrity gate

The confirmatory run reads two small files and nothing else: `labels.csv`, which
carries the 79 attack phase windows, and `ait_corpus_v1.json`, which carries the
derived case packets and the P2 answer key. Both are checksummed here and both
digests enter the design hash in Cell 9.

The AIT alert archive is **not** staged here and must not be reachable from this
notebook. It holds 2,655,821 alerts across sixteen JSON files and is opened
exactly once, by the separate derivation notebook. That separation is not
housekeeping. A run that can reach the raw archive can redraw a packet after a
result has been seen, and nothing in the artefact would record that it happened.
Pinning a digest on a small derived file does not prevent a redraw, it makes one
visible.

Under `PILOT` this cell warns and continues, because the pilot runs on the four
synthetic cases. Under a confirmatory run it raises unless both files are
present, both digests are pinned, both match, and the corpus carries the hash of
the sampling rule that produced it.

In [ ]:
# =====================================================================
# Cell 1b: data staging and integrity gate
# ---------------------------------------------------------------------
# PLACEMENT, corrected. This was drafted as Cell 0b. Cell 0 is a bare
# !rm -rf that runs before a single import exists, so a staging cell
# placed there could not hash a file, could not read PILOT and could
# not use Path. It sits after Cell 1 for the same reason 3b sits
# after 3. Nothing is renumbered.
#
# WHAT THE CONFIRMATORY RUN IS ALLOWED TO READ.
# Exactly two files, both small:
#   labels.csv          3,703 bytes, 79 rows, the attack phase windows
#   ait_corpus_v1.json  the derived case packets and the P2 answer key
#
# It does NOT read the AIT alert archive. That archive holds 2,655,821
# alerts across sixteen JSON files. It is source material, not runtime
# data. It is opened exactly once, by the separate derivation notebook,
# which is deliberately not part of this pipeline and must not be run
# inside it. Two reasons.
#   Practical: no scenario file may be loaded whole. wheeler alone
#   carries 616,161 alerts.
#   Methodological, and the more important of the two: if the archive
#   is reachable from the confirmatory run, a packet can be redrawn
#   after a result has been seen. Pinning a checksum on a small derived
#   file does not make that impossible, it makes it detectable. That is
#   the most a pre-registration can ever do.
#
# FAIL CLOSED. Under PILOT this cell warns. Under a confirmatory run it
# raises unless both files are present, both checksums are pinned in
# this cell, both match, and the corpus carries the hash of the
# sampling rule that produced it. A staging step that quietly proceeds
# on a missing or changed corpus is worse than no staging step, because
# every later number would still be produced and would still look fine.
# =====================================================================

AIT_ZENODO_RECORD = '8263181'
AIT_ZENODO_API    = 'https://zenodo.org/api/records/' + AIT_ZENODO_RECORD
AIT_ZENODO_HTML   = 'https://zenodo.org/records/' + AIT_ZENODO_RECORD

AIT_LABELS_PATH = '/content/labels.csv'
AIT_CORPUS_PATH = '/content/ait_corpus_v1.json'

# PIN THESE after the derivation notebook prints them. Leave as None
# only while PILOT is True. The derivation notebook prints both lines
# ready to paste, so there is no reason to transcribe a digest by hand.
AIT_LABELS_SHA256 = '633e09b42772eabbbe5b48598220860eead60b7352efb01973fc45a5034290c0'
AIT_CORPUS_SHA256 = '16eb5b403b9e47a11dd8cdddf5884b53ba6f3818c7d6b1f72d86b691c12e656f'

# Set from the corpus metadata below. Enters the design hash in Cell 9.
SAMPLING_RULE_HASH = None
AIT_CORPUS_META    = {}

# CC BY 4.0 section 3(a) requires attribution and an indication that
# the material was modified. This string travels with every derived
# case in its provenance field and is not optional.
AIT_ATTRIBUTION = (
    'Derived from the AIT Alert Data Set, Landauer et al., Zenodo record '
    + AIT_ZENODO_RECORD + ', ' + AIT_ZENODO_HTML + ', licensed CC BY 4.0. '
    'Modified: alerts were filtered to bounded time windows, timestamps were '
    'shifted by a fixed offset, scenario names were replaced, and only '
    'extracts appear in the derived case packets. No AIT-ADS repository code '
    'was copied, so the GPL-3.0 on that repository is not engaged.'
)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def copy_from_drive(name, subdir='AXIOM/AIT'):
    """Convenience for the repeat case. The Colab upload panel loses the
    file on every disconnect, and re-uploading by hand is the step most
    likely to be skipped or done from the wrong copy."""
    from google.colab import drive
    if not Path('/content/drive').exists():
        drive.mount('/content/drive')
    src = Path('/content/drive/MyDrive') / subdir / name
    if not src.exists():
        raise SystemExit('not in Drive: ' + str(src))
    dst = Path('/content') / name
    dst.write_bytes(src.read_bytes())
    print('staged', name, 'from Drive,', dst.stat().st_size, 'bytes')
    return str(dst)


def stage_check():
    """Report on both required files, then gate. Returns a DataFrame so
    the state is visible rather than inferred from an absence of
    errors."""
    global SAMPLING_RULE_HASH, AIT_CORPUS_META
    wanted = [('labels.csv', AIT_LABELS_PATH, AIT_LABELS_SHA256),
              ('ait_corpus_v1.json', AIT_CORPUS_PATH, AIT_CORPUS_SHA256)]
    rows, problems = [], []
    for name, path, pinned in wanted:
        p = Path(path)
        if not p.exists():
            rows.append({'file': name, 'present': False, 'bytes': 0,
                         'sha256': '', 'pinned': bool(pinned),
                         'match': False})
            problems.append(name + ' is not at ' + path)
            continue
        got = sha256_file(path)
        match = (pinned == got) if pinned else None
        rows.append({'file': name, 'present': True, 'bytes': p.stat().st_size,
                     'sha256': got[:16] + '...', 'pinned': bool(pinned),
                     'match': match})
        if pinned and not match:
            problems.append(
                name + ' does not match its pinned digest. Expected '
                + pinned[:16] + '..., got ' + got[:16] + '.... Either the '
                'file changed or the pin is stale. Do not proceed on either.')
        if not pinned:
            problems.append(name + ' has no pinned digest in Cell 1b')

    if Path(AIT_CORPUS_PATH).exists():
        try:
            AIT_CORPUS_META = json.loads(
                Path(AIT_CORPUS_PATH).read_text()).get('derivation', {})
        except Exception as e:
            problems.append('corpus is not readable JSON: ' + str(e))
        SAMPLING_RULE_HASH = AIT_CORPUS_META.get('sampling_rule_hash')
        if not SAMPLING_RULE_HASH:
            problems.append(
                'the corpus carries no sampling_rule_hash, so there is no '
                'evidence the draw was made under a rule fixed in advance. '
                'Regenerate it with the derivation notebook.')

    df = pd.DataFrame(rows)
    display(df)
    print('zenodo record  ', AIT_ZENODO_HTML, '(CC BY 4.0)')
    print('sampling rule  ', SAMPLING_RULE_HASH or 'NOT SET')
    print('derived        ', AIT_CORPUS_META.get('derived_utc', 'n/a'),
          '| packets', AIT_CORPUS_META.get('n_packets', 0))

    if problems:
        msg = 'data staging incomplete:' + NL + '  - ' + (NL + '  - ').join(problems)
        if PILOT:
            print(NL + 'WARNING under PILOT, not fatal:' + NL + msg)
            print(NL + 'The pilot runs on the four synthetic cases, so it '
                  'does not need the AIT corpus. A confirmatory run does, '
                  'and this cell will refuse to pass until every line above '
                  'is resolved.')
        else:
            raise SystemExit(msg)
    else:
        print(NL + 'staging OK, both inputs present and pinned')
    return df


_stage = stage_check()

# Step 02 · Cell 2: models, provider lock


In [ ]:
# Model ids carry the provider suffix when routing through HF.
PROVIDER_SUFFIX = ':deepinfra'

# MODELS is the GENERATOR set. The judge is deliberately outside it: a model
# that appears in both scores its own outputs on part of the corpus, which the
# old guard did not catch because it only excluded qwen by name.
# Selected 2026-08-04 from the deepinfra capability table. Three distinct
# lineages, all with supports_structured_output true, all under $0.60 per 1M
# output tokens. Capability is deliberately unequal: if the harness only helps
# the strongest model, that is a finding, not a nuisance.
# Qwen3-235B-A22B-Instruct-2507 REPLACED 2026-08-04 after pilot 3. On this
# endpoint it returned one word per line with blank lines between, and in one
# trial looped the word 'The' to the token cap. Every such output scored as a
# parse failure, spread across all three arms, and its prose abstentions under
# arm B were the behaviour that arm exists to elicit. Replaced rather than
# debugged: this is a study of harnesses, not of serving configurations. The
# substitution is made before any confirmatory data exists and for a reason
# recorded in the pilot 3 readout, which is the only kind of model change this
# design permits.
# gemma-3-27b-it keeps three distinct lineages, now Google, DeepSeek and Meta,
# at $0.08 in and $0.16 out, cheaper than the model it replaces on output by a
# factor of three.
MODELS = {
    'gemma'    : 'google/gemma-3-27b-it' + PROVIDER_SUFFIX,
    'deepseek' : 'deepseek-ai/DeepSeek-V4-Flash-0731' + PROVIDER_SUFFIX,
    'llama'    : 'meta-llama/Llama-4-Scout-17B-16E-Instruct' + PROVIDER_SUFFIX,
}

# Judge. GLM-4.6 rather than GLM-5.2: verified working, leak-free and cheaper.
# Judge continuity with v0.3 buys nothing, since the corpus and the unit of
# analysis both differ.

# Excluded on purpose, recorded so the exclusions enter the design hash:
#   meta-llama/Llama-3.3-70B-Instruct  400 model_not_supported on deepinfra.
#   *-Thinking-*, DeepSeek-R1-*  reasoning-first. enable_thinking is unreliable
#     and leaked <think> tags would fail the parser at a rate unrelated to the
#     harness under test.
#   *-VL-*  vision variants, no visual input in this corpus.
#   microsoft/phi-4  16k context, the smallest available. Arm G discloses all
#     three tiers, so context headroom must not differ by model.
#   Qwen/Qwen3.6-27B  v0.3's generator, but $3.20 per 1M output, roughly six
#     times Qwen3-235B. Generator continuity with v0.3 buys nothing here
#     because the corpus and unit of analysis are different.
#   openai/gpt-oss-*  standing preference against OpenAI-derived models.
#   All zai-org/GLM-*  reserved to the judge role, must not generate.
#   Qwen/Qwen3-235B-A22B-Instruct-2507  degenerate output on this endpoint,
#     see the pilot 3 readout. Excluded on measured serving behaviour before
#     any confirmatory data existed, not on a result it produced.
#   Qwen/Qwen3.5-9B, Qwen/Qwen3-30B-A3B  same lineage as the model just
#     removed. If the defect is lineage or template level rather than specific
#     to that id, a sibling would reproduce it at a lower and quieter rate.
#   nvidia/NVIDIA-Nemotron-3-Ultra-550B  $5.00 per 1M output, ten times the
#     replacement, and the judge already sits at $2.00.

JUDGE_MODEL = 'zai-org/GLM-4.6' + PROVIDER_SUFFIX

# Models whose chat template accepts enable_thinking. The kwarg was previously
# sent to every model, including instruct-only ids with no thinking mode at
# all, and that is a live suspect for the qwen defect. It is now sent only
# where it means something. The <think> leak check in preflight still runs on
# every model regardless, because suppression failing silently is worse than
# the kwarg being rejected loudly.
THINKING_CAPABLE = {'zai-org/GLM-4.6', 'zai-org/GLM-5.2'}

# CrowdStrike exclusion is an employment constraint. No CrowdStrike-derived
# model or dataset in this run. Recorded here so it appears in the hash.
EXCLUDED_VENDORS = ['crowdstrike']

PROVIDER_MAP = {m: m.split(':')[-1] for m in MODELS.values()}

def assert_providers():
    bad = [m for m, p in PROVIDER_MAP.items() if p != 'deepinfra']
    if bad:
        raise SystemExit(f'provider mismatch, refusing to run: {bad}')
    if JUDGE_MODEL in MODELS.values():
        raise SystemExit('judge model is also a generator. It would score its '
                         'own outputs on part of the corpus.')
    if 'router.huggingface.co' in BASE_URL and not all(
            ':' in m for m in MODELS.values()):
        raise SystemExit('HF router requires a provider suffix on every model id')
    if 'api.deepinfra.com' in BASE_URL and any(':' in m for m in MODELS.values()):
        raise SystemExit('direct DeepInfra calls must not carry a provider suffix')
    if len(set(MODELS.values())) != len(MODELS):
        raise SystemExit('duplicate generator ids: one model would be counted '
                         'as two lineages')
assert_providers()
print('providers locked')

providers locked


# Step 03 · Cell 3: client


In [ ]:
from openai import OpenAI
from tenacity import (retry, stop_after_attempt, wait_exponential,
                      retry_if_not_exception_type)

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
PERMANENT_CODES = {400, 401, 403, 404, 422}

class PermanentError(Exception): pass

# CORRECTED 2026-08-04. The previous decorator used
# retry_if_exception_type(Exception), which caught PermanentError and retried
# a permanent 401 five times with exponential backoff. PERMANENT_CODES only
# short-circuits if the retry predicate excludes PermanentError.
@retry(stop=stop_after_attempt(5),
       wait=wait_exponential(min=2, max=60),
       retry=retry_if_not_exception_type(PermanentError),
       reraise=True)
def _call(model, messages, max_tokens, temperature):
    # Only sent to ids that have a thinking mode. See THINKING_CAPABLE.
    extra = ({'chat_template_kwargs': {'enable_thinking': False}}
             if model.split(':')[0] in THINKING_CAPABLE else {})
    try:
        r = client.chat.completions.create(
            model=model, messages=messages,
            max_tokens=max_tokens, temperature=temperature,
            extra_body=extra,
        )
    except Exception as e:
        code = getattr(getattr(e, 'response', None), 'status_code', None)
        if code in PERMANENT_CODES:
            raise PermanentError(f'{code}: {e}') from e
        raise
    return r

def run_one(model, system, user, max_tokens=1200, temperature=0.7):
    """Returns (text, usage_dict). Token usage is recorded on every call
    because method rule 6 requires length alongside every quality metric."""
    msgs = [{'role': 'system', 'content': system},
            {'role': 'user',   'content': user}]
    r = _call(model, msgs, max_tokens, temperature)
    u = r.usage
    return r.choices[0].message.content, {
        'prompt_tokens': u.prompt_tokens,
        'completion_tokens': u.completion_tokens,
        'total_tokens': u.total_tokens,
        # finish_reason == 'length' means the output was cut off. Truncation
        # produces unparseable JSON, and the AXIOM harness demands the longest
        # object of any arm, so truncation is expected to correlate with arm.
        # Untracked, it would appear as arm E failing to answer.
        'finish_reason': r.choices[0].finish_reason,
    }

# Step 04 · Cell 3b: selection check


In [ ]:
# Selected 2026-08-04:
#   generators  gemma-3-27b-it, DeepSeek-V4-Flash-0731,
#               Llama-4-Scout-17B-16E-Instruct
#   judge       GLM-4.6
# gemma-3-27b-it replaced Qwen3-235B-A22B-Instruct-2507 after pilot 3. It has
# not yet passed preflight. Cell 4 is the qualification, and it will refuse to
# proceed on degenerate output or a failed JSON round trip.
# Rejected candidates and the reasons are recorded in the Cell 2 comment block
# so the exclusions survive in the design hash rather than only in chat.

def check_selection(provider='deepinfra'):
    """Confirm the hashed model ids are still live and still support structured
    output, and surface current prices. Not a search: it can only pass or fail
    on the models already committed to."""
    import requests
    r = requests.get(f'{BASE_URL}/models',
                     headers={'Authorization': f'Bearer {API_KEY}'}, timeout=60)
    r.raise_for_status()
    meta = {}
    for m in r.json().get('data', []):
        for p in m.get('providers', []):
            if p.get('provider') == provider and p.get('status') == 'live':
                meta[m['id']] = p
    rows = []
    for role, mid in list(MODELS.items()) + [('judge', JUDGE_MODEL)]:
        base = mid.split(':')[0]
        p    = meta.get(base)
        pr   = (p or {}).get('pricing') or {}
        rows.append({'role': role, 'id': base, 'live': p is not None,
                     'structured': (p or {}).get('supports_structured_output'),
                     'ctx': (p or {}).get('context_length'),
                     'tok_s': (p or {}).get('throughput'),
                     'in_$': pr.get('input'), 'out_$': pr.get('output')})
    df = pd.DataFrame(rows)
    display(df)
    if not df.live.all():
        raise SystemExit(f'no longer served on {provider}: {list(df[~df.live].id)}')
    if not df.structured.all():
        raise SystemExit('a selected model no longer reports structured output '
                         'support. Parse failures would follow, unrelated to arm.')
    return df

sel = check_selection()

,role,id,live,structured,ctx,tok_s,in_$,out_$
0,gemma,google/gemma-3-27b-it,True,True,131072,33.604900,0.08,0.16
1,deepseek,deepseek-ai/DeepSeek-V4-Flash-0731,True,True,1048576,53.283993,0.09,0.18
2,llama,meta-llama/Llama-4-Scout-17B-16E-Instruct,True,True,327680,44.382327,0.10,0.30
3,judge,zai-org/GLM-4.6,True,True,202752,53.213565,0.50,2.00


In [ ]:
# Candidate probing removed 2026-08-04, selection closed. The four ids above
# are the only models this study may use. Changing one after results are seen
# requires a new DESIGN_HASH and a stated reason, not an edit to this cell.
assert len(MODELS) == 3 and JUDGE_MODEL not in MODELS.values()
print('selection locked:', list(MODELS.values()), '| judge', JUDGE_MODEL)

selection locked: ['google/gemma-3-27b-it:deepinfra', 'deepseek-ai/DeepSeek-V4-Flash-0731:deepinfra', 'meta-llama/Llama-4-Scout-17B-16E-Instruct:deepinfra'] | judge zai-org/GLM-4.6:deepinfra


In [ ]:
# Full-catalogue browsing removed. If a selected model is ever withdrawn,
# check_selection() fails loudly and the replacement is a deliberate,
# re-hashed decision rather than a convenient one made mid-analysis.

# Step 05 · Cell 4: preflight


In [ ]:
def is_degenerate(t):
    """Added 2026-08-04 from the pilot 3 diagnostics. Qwen returned outputs of
    the form 'the / evidence / provided / does / not / support' one word per
    line, and in one case looped the word 'The' to the 4096 token cap. Every
    such output was counted as a parse failure, so a serving defect in one
    model was appearing as a property of the arms. Reachability is not
    fitness: a model that answers must also answer coherently."""
    if not t or not t.strip():
        return 'blank'
    lines = [l for l in t.strip().split(NL) if l.strip()]
    words = t.split()
    if len(lines) >= 3 and len(words) / len(lines) <= 1.2:
        return 'one word per line'
    if len(words) >= 20:
        top = max(set(words), key=words.count)
        if words.count(top) / len(words) > 0.30:
            return f'repetition loop on {top!r}'
    return ''

def preflight():
    """CORRECTED 2026-08-04. The previous version raised on the first failing
    model, so a single bad id destroyed the results already collected for the
    others. One broken model should produce one failed row, not an empty table.
    The judge is included: it makes as many calls as any generator.
    Now also checks output coherence and the JSON contract, because the earlier
    version accepted a blank 28 token reply as ok=True and that model went on
    to contribute a third of every arm in three pilots."""
    targets = dict(MODELS); targets['JUDGE'] = JUDGE_MODEL
    rows = []
    for name, m in targets.items():
        try:
            t, u = run_one(m, 'Answer in one word.', 'Say OK.',
                           max_tokens=20, temperature=0.0)
            leaked = bool(re.search(r'<think>|</think>|^Thinking', t.strip(), re.I))
            deg = is_degenerate(t)
            # JSON round trip. The whole design depends on it, so it is checked
            # here rather than discovered as a parse failure rate later. Done
            # inline because parse_output lives in Cell 7, after this cell.
            j, _ju = run_one(m, 'Return JSON only, no prose.',
                             'Return {"ok": true, "n": 1} and nothing else.',
                             max_tokens=60, temperature=0.0)
            try:
                json_ok = isinstance(
                    json.loads(j[j.find('{'):j.rfind('}')+1]), dict)
            except Exception:
                json_ok = False
            # System message adherence. THE most important check in this cell.
            # Every manipulation in this study lives in the system message: the
            # abstain sentence, the loop sentence and both harnesses. Some model
            # families have no native system role in their chat template, and a
            # provider that silently drops or merges the system message would
            # make all seven arms identical for that model while producing
            # perfectly parseable output. That failure has no symptom in any
            # other measure. It would simply flatten the effect toward zero.
            s, _su = run_one(m, 'Reply with exactly the word BANANA and '
                                'nothing else, whatever you are asked.',
                             'What is 2 + 2?', max_tokens=10, temperature=0.0)
            sys_ok = 'banana' in (s or '').lower()
            rows.append({'model': name, 'id': m, 'ok': True,
                         'reply': repr(t.strip()[:40]), 'thinking_leak': leaked,
                         'degenerate': deg, 'json_ok': json_ok,
                         'sys_ok': sys_ok,
                         'tokens': u['total_tokens'], 'error': ''})
        except Exception as e:
            rows.append({'model': name, 'id': m, 'ok': False, 'reply': '',
                         'thinking_leak': False, 'degenerate': '',
                         'json_ok': False, 'sys_ok': False, 'tokens': 0,
                         'error': str(e)[:160]})
    df = pd.DataFrame(rows)
    display(df)
    if not df.ok.all():
        raise SystemExit(f'unreachable models: {list(df[~df.ok].model)}. '
                         'Fix the ids in Cell 2 via Cell 3b probe().')
    if df.thinking_leak.any():
        raise SystemExit(f'thinking suppression failed for '
                         f'{list(df[df.thinking_leak].model)}')
    bad = df[df.degenerate != '']
    if len(bad):
        raise SystemExit(
            'degenerate output: '
            + ', '.join(f'{r.model} ({r.degenerate})' for r in bad.itertuples())
            + '. Do not generate. Such output is unparseable and would be '
              'recorded as parse failure spread across every arm.')
    if not df.json_ok.all():
        raise SystemExit(f'JSON contract failed for '
                         f'{list(df[~df.json_ok].model)}. Every measure in this '
                         'study reads from parsed JSON.')
    if not df.sys_ok.all():
        raise SystemExit(
            f'system message ignored or dropped for {list(df[~df.sys_ok].model)}. '
            'Every arm manipulation lives in the system message, so for that '
            'model all seven arms would be the same prompt. This would not show '
            'up as an error anywhere else in the pipeline: it would show up as '
            'no effect.')
    if len(MODELS) < 2:
        raise SystemExit('fewer than two generator models. A single-generator '
                         'result cannot separate framework effects from model '
                         'idiosyncrasy. Add a second before generating.')
    return df

preflight()

,model,id,ok,reply,thinking_leak,degenerate,json_ok,sys_ok,tokens,error
0,gemma,google/gemma-3-27b-it:deepinfra,True,'OK.',False,,True,True,22,
1,deepseek,deepseek-ai/DeepSeek-V4-Flash-0731:deepinfra,True,'OK.',False,,True,True,15,
2,llama,meta-llama/Llama-4-Scout-17B-16E-Instruct:deep...,True,'OK',False,,True,True,25,
3,JUDGE,zai-org/GLM-4.6:deepinfra,True,'OK',False,,True,True,21,


,model,id,ok,reply,thinking_leak,degenerate,json_ok,sys_ok,tokens,error
0,gemma,google/gemma-3-27b-it:deepinfra,True,'OK.',False,,True,True,22,
1,deepseek,deepseek-ai/DeepSeek-V4-Flash-0731:deepinfra,True,'OK.',False,,True,True,15,
2,llama,meta-llama/Llama-4-Scout-17B-16E-Instruct:deep...,True,'OK',False,,True,True,25,
3,JUDGE,zai-org/GLM-4.6:deepinfra,True,'OK',False,,True,True,21,


# Step 06 · Cell 5: cases and reference keys


In [ ]:
# resolving_field  : the field that decides the case. None if unresolvable.
# resolvable       : does that field exist anywhere in tiers 1 or 2
# flip_checks      : checks that COULD change the verdict. Used to score
#                    whether a model's falsifier is discriminating rather
#                    than ritual. A falsifier naming a check that could
#                    never fire scores zero.

# resolving_aliases : the ways an analyst might legitimately name the
#                     resolving field. Authored with the case, before any
#                     model output exists, so the matcher cannot be tuned
#                     to the outputs it scores.

CASE_SCHEMA = ['case_id','case_type','tier0','tier1','tier2','resolving_field',
               'resolving_aliases','resolvable','true_verdict','true_cause',
               'flip_checks','provenance']

# Windows path separators halved 2026-08-04. Every path in this cell was
# written with four backslashes in the source, which is two in the string the
# model actually sees, so every disclosed path was doubled. A model that quoted
# a path in its ordinary single separator form failed verify_quotes and was
# recorded as fabricating: 62 of 64 quote failures in the pilot 5 rescore.
# Single escaping is also simply correct, since these are meant to read as real
# telemetry. This changes CASES, so TRIAL_FINGERPRINT changes and the existing
# pilot trials must be regenerated rather than rescored.
CASES = [
{
 'case_id':'N01', 'case_type':'clean_null',
 'tier0': textwrap.dedent("""\
   ALERT: EDR-1188 Suspicious PowerShell
   host: WKS-4471   user: svc_backup   time: 2026-07-14T02:14:03Z
   process: powershell.exe -NoProfile -ExecutionPolicy Bypass -File C:\\Scripts\\vss_snap.ps1
   parent: taskeng.exe   pid: 4820   ppid: 812
   network: none observed in 60m window
   file writes: D:\\Backups\\vss_20260714.log
   """),
 'tier1': textwrap.dedent("""\
   SCHEDULED TASK: \\Microsoft\\Backup\\VSSNightly  created 2024-03-02 by DOMAIN\\adm_infra
   task runs daily 02:14. last 90 executions all succeeded.
   script hash sha256 4c1f...9ab2 present in approved-script inventory since 2024-03-02
   """),
 'tier2': 'PROXY: no egress from WKS-4471 in 24h window. AUTH: svc_backup no interactive logons in 90d.',
 'resolving_field':'scheduled_task_provenance',
 'resolving_aliases':['scheduled task','scheduled_task','task provenance',
                      'task creation','task history','approved script',
                      'script inventory','script hash','execution history'],
 'resolvable':True,
 'true_verdict':'benign', 'true_cause':'scheduled backup task, approved script, stable 90 day baseline',
 'flip_checks':['script hash absent from approved inventory',
                'task created within 7 days of alert',
                'egress observed from host in window'],
 'provenance':'synthetic, authored 2026-08-04',
},
{
 'case_id':'U01', 'case_type':'underdetermined',
 'tier0': textwrap.dedent("""\
   ALERT: NET-2210 Beaconing pattern to 185.234.219.x
   host: LAP-0912   first seen 2026-07-22T09:04Z   interval 300s +/- 4s   42 connections
   bytes out 1.1KB avg, bytes in 340B avg
   process attribution: UNAVAILABLE (endpoint agent offline 2026-07-20 to 2026-07-24)
   """),
 'tier1': 'THREATINTEL: 185.234.219.0/24 mixed reputation. 3 of 70 engines flag. no current campaign attribution.',
 'tier2': 'ASSET: LAP-0912 last EDR checkin 2026-07-20T11:02Z. No process telemetry retained for the window.',
 'resolving_field':'process_attribution',
 'resolving_aliases':['process attribution','process_attribution','process name',
                      'process telemetry','process id','endpoint agent',
                      'edr telemetry','command line'],
 'resolvable':False,
 'true_verdict':'insufficient',
 'true_cause':'regular beaconing with no process attribution available. Cannot distinguish updater from C2. Escalate for collection.',
 'flip_checks':['process attribution recovered from netflow-adjacent source',
                'identical pattern found on hosts with agent coverage'],
 'provenance':'synthetic, authored 2026-08-04',
},
{
 'case_id':'A01', 'case_type':'adversarial_null',
 'tier0': textwrap.dedent("""\
   ALERT: EDR-3402 Remote management tool execution
   host: WKS-2210   user: jsomers   time: 2026-07-18T14:41:52Z
   process: ScreenConnect.ClientService.exe
   binary path: C:\\Program Files (x86)\\ScreenConnect Client\\
   sha256 7e02...c41d   VirusTotal 3 of 71 engines flag, all heuristic RiskWare
   network: outbound TLS to 104.36.84.0/24, 41m session
   note: destination range appears on internal blocklist entry BL-0912
   """),
 'tier1': textwrap.dedent("""\
   CHANGE MGMT: CHG-4471 approved 2025-11-03, helpdesk RMM rollout, 812 endpoints
   ASSET: WKS-2210 in scope for CHG-4471. Agent installed 2025-11-19.
   BLOCKLIST: BL-0912 created 2023-06-02 for an unrelated hosting tenant,
     no review since. 104.36.84.0/24 reassigned to the RMM vendor 2024-08.
   """),
 'tier2': ('TICKET: HD-88214 open at 14:38, user reported printer fault, '
           'closed 15:22 by helpdesk operator kbrennan. Session duration matches.'),
 'resolving_field':'rmm_authorisation_record',
 'resolving_aliases':['change management','change record','change window','chg-',
                      'authorisation','authorization','approved tool',
                      'software policy','deployment record','helpdesk ticket',
                      'support ticket','ticket id','sanctioned'],
 'resolvable':True,
 'true_verdict':'benign',
 'true_cause':'authorised helpdesk RMM session during an open support ticket. '
              'Blocklist hit is a stale entry against reassigned address space.',
 'flip_checks':['installation date outside the CHG-4471 rollout window',
                'binary hash differs from the vendor signed release',
                'session initiated with no corresponding helpdesk ticket',
                'remote session source outside the helpdesk address range'],
 'provenance':'synthetic, authored 2026-08-04',
},
{
 'case_id':'P01', 'case_type':'true_positive',
 'tier0': textwrap.dedent("""\
   ALERT: EDR-5517 rundll32 executing unusual export
   host: FIN-0331   user: mchen   time: 2026-07-25T11:07:19Z
   process: rundll32.exe C:\\Users\\mchen\\AppData\\Local\\Temp\\ui32.dll,DllRegisterServer
   parent: UNRESOLVED in alert payload
   sha256 not present in any reputation source
   network: not included in this alert payload
   """),
 'tier1': textwrap.dedent("""\
   PROCESS TREE: winword.exe -> cmd.exe -> rundll32.exe
     winword.exe opened Invoice_Q3_revised.docm from Outlook temp path
   PROXY: FIN-0331 -> cdn-analytics-eu[.]info 11:07:44Z onward,
     regular 60s intervals, TLS, 118 connections
   DOMAIN: cdn-analytics-eu[.]info registered 2026-07-19, six days before alert
   """),
 'tier2': textwrap.dedent("""\
   AUTH: FIN-0331 -> SRV-FS02 SMB, account mchen, 11:22Z, first ever occurrence
   LSASS: handle open from rundll32.exe 11:14Z, access mask 0x1010
   """),
 'resolving_field':'parent_process_chain',
 'resolving_aliases':['parent process','parent_process','parent image',
                      'process tree','process chain','process ancestry','ppid'],
 'resolvable':True,
 'true_verdict':'malicious',
 'true_cause':'macro-borne loader from a mailed document executing via rundll32, '
              'beaconing to a newly registered domain, followed by credential '
              'access and first-time lateral SMB',
 'flip_checks':['parent chain resolves to a signed software installer',
                'domain has multi-year registration and prior enterprise usage',
                'DLL export matches a known vendor product',
                'SMB access to SRV-FS02 is routine for this account'],
 'provenance':'synthetic, authored 2026-08-04',
},
# TODO N02..N12 clean_null, A02..A06 adversarial_null,
#      P02..P08 true_positive, U02..U06 underdetermined
]

def validate_cases(cases):
    errs = []
    for c in cases:
        for k in CASE_SCHEMA:
            if k not in c: errs.append(f"{c.get('case_id','?')} missing {k}")
        if c.get('case_type') == 'underdetermined' and c.get('resolvable'):
            errs.append(f"{c['case_id']} underdetermined but marked resolvable")
        if c.get('resolvable') and not c.get('resolving_field'):
            errs.append(f"{c['case_id']} resolvable but no resolving_field")
        if not c.get('flip_checks'):
            errs.append(f"{c['case_id']} no flip_checks, falsifier cannot be scored")
        if c.get('resolving_field') and not c.get('resolving_aliases'):
            errs.append(f"{c['case_id']} resolving_field with no aliases, "
                        'collection_targeting would score on one arbitrary word')
    return errs

_e = validate_cases(CASES)
print('case validation:', 'OK' if not _e else _e)
print(f'{len(CASES)} cases authored, 32 required')

# Step 07 · Cell 5b: corpus contamination probe


# Step 07 · Cell 5b: corpus contamination probe

A gate, not a diagnostic. It runs after Cell 5 because it needs `CASES`,
`MODELS`, `JUDGE_MODEL` and `run_one`, and before Cell 8 costing and the
Cell 9 hash, because a corpus the generators can answer from memory cannot
measure an abstain-and-request loop at all.

Four arms. P1 documentation recall. P2 data recall. P3 the same twenty
question forms asked of KRONOS-7, a corpus that does not exist, which sets
the confabulation floor and is the thing that makes this an instrument
rather than an anecdote. P4 case level, anchored versus de-anchored, the
only arm that can reject the source. Correct abstention scores as correct.
Recall is a maximum over samples, never a mean.

The judge is probed alongside the three generators. A contaminated judge
grades `cause_match` against memory rather than against the provided
reference, silently and consistently, so the error would not look like
noise.

**De-anchoring, revised 2026-08-05.** `labels.csv` confirmed that timestamps
are absolute Unix epoch seconds, so a uniform offset is a viable operation.
It is also insufficient on its own. All eight scenarios run the same ten
phase chain in near identical order with tightly clustered durations, so the
fingerprint of this corpus is its phase structure, which a uniform offset
leaves untouched. The fix is `packet_window_ok()`: a P4 packet must lie
wholly inside one labelled phase, or wholly inside one benign interval. A
packet that spans a boundary carries the chain ordering and is redrawn.
Jittering the intervals was rejected, because inconsistent timing is exactly
what a triage packet must not contain.

**Guard band, fixed 2026-08-05 at 60s leading and 600s trailing.** It applies
to the **null draw only**. Out of window is a false positive by construction
and is the only label here that needs no human judgement, whereas in window
does not make an alert a true positive, so the band protects the null side and
has no job on the positive side. It is asymmetric because detection lags
generation and never precedes it: Suricata fires at packet time, Wazuh at log
write time, and AMiner aggregates over windows, so the trailing value must be
the generous one. Cost is under one percent of benign time across the corpus.

An earlier draft applied one symmetric 300s band to both sides. Measured
against `labels.csv` that was destructive: only 23 of 79 phases were long
enough to hold an interior packet, and `reverse_shell`,
`privilege_escalation`, `service_stop`, `webshell` and `wpscan` went to zero.
The positive set would have collapsed to `cracking` and `dnsteal` and every
fast decisive action in the kill chain would have vanished silently.

On the first run this cell is a **negative control**. `PROBE_CASES` defaults
to `CASES`, which holds four synthetic cases authored on 2026-08-04.
Synthetic cases cannot be contaminated, so anchored and de-anchored accuracy
should match. A gap means the machinery is broken, not that the source is
dirty. It becomes a verdict on AIT only once `AIT_PACKETS` is filled by the
deterministic sampling rule.


In [ ]:
# ==========================================================================
# CELL 5b. CORPUS CONTAMINATION PROBE
# ==========================================================================
# Written 2026-08-05. Runs after Cell 5 and before the Cell 8 costing and the
# Cell 9 hash. It needs run_one (Cell 3), MODELS and JUDGE_MODEL (Cell 2) and
# CASES (Cell 5). Nothing later in the notebook is required. This cell does
# not call parse_output_loose, which lives in Cell 12: the small JSON reader
# below exists solely to avoid a forward reference.
#
# WHY CONTAMINATION MATTERS DIFFERENTLY HERE.
# In an ordinary benchmark contamination inflates a score and you discount it.
# Here it deletes the mechanism under test. The abstain-and-request loop only
# exists if the model genuinely lacks information. A model that already knows
# the answer has no reason to request tier 1, so gate_correctness,
# collection_targeting and fp_rate_on_nulls all stop measuring what they claim,
# and arm E would look disciplined while actually performing a lookup.
#
# THREE LEVELS OF EXPOSURE, ONLY THE THIRD IS FATAL.
#   L1  the dataset exists, its documentation, its scenario names. Harmless.
#   L2  aggregate facts: alert counts, engine split, attack list. Near
#       harmless, because none of it resolves a single case.
#   L3  case-level ground truth from a tier 0 packet alone. FATAL. The source
#       cannot support the loop and is rejected outright.
#
# FOUR ARMS.
#   P1  documentation recall, answerable from the Zenodo record or the README
#   P2  data recall, answerable only from inside the files
#   P3  the same twenty question forms against a corpus that does not exist.
#       This is what makes the probe an instrument rather than an anecdote. It
#       sets the confabulation floor, and P1 and P2 count as recall only to the
#       extent that they exceed it. Without P3 this cell would be exactly the
#       null-free evaluation that method rule 1 forbids.
#   P4  case level, anchored versus de-anchored. The only arm that can reject
#       the source.
#
# SCORING RULE THAT MUST NOT BE GOT WRONG.
# A model replying 'I do not have access to that dataset' is ANSWERING
# CORRECTLY and is scored as a correct abstention. Marking it wrong would
# penalise the exact behaviour this project values, and would import into the
# probe the bias the framework exists to remove.
#
# TWO STATED LIMITS.
#   1. The result is valid only for the pinned model versions on the run date.
#      A model swap invalidates it and the probe must be rerun.
#   2. Publishing the corpus contaminates future models, so the corpus has a
#      shelf life and any post-publication replication needs its own probe.

from scipy.stats import binomtest

PROBE_RESULT = None          # Cell 9 refuses to hash while this is None
PROBE_RUN_DATE = None        # set by run_probe(), enters PREREG

PROBE_SOURCE = {
    'name'     : 'AIT Alert Data Set (AIT-ADS)',
    'record'   : 'https://zenodo.org/records/8263181',
    'licence'  : 'CC BY 4.0, Zenodo data record, verified 2026-08-05',
    'scenarios': ['fox', 'harrison', 'russellmitchell', 'santos',
                  'shaw', 'wardbeck', 'wheeler', 'wilson'],
}

# The fabricated control. Same shape as the real source, bird names rather
# than surnames, a plausible but non-existent Zenodo record. Every question
# form asked of AIT is asked of this. Any confident answer is confabulation.
PROBE_DECOY = {
    'name'     : 'KRONOS-7 SOC Alert Corpus (2024)',
    'record'   : 'https://zenodo.org/records/7741190',
    'licence'  : 'n/a, this corpus does not exist',
    'scenarios': ['kestrel', 'merlin', 'goshawk', 'harrier',
                  'osprey', 'peregrine', 'saker', 'hobby'],
}

# Recall is scored as a MAXIMUM over samples, never a mean. One correct answer
# in five proves the fact is in the weights; four wrong answers do not unprove
# it. Sampling at the experiment's own temperature rather than at zero, because
# the question is what the model can produce, not what it usually produces.
PROBE_SAMPLES = 5
PROBE_TEMPERATURE = 0.7

PROBE_SYSTEM = ('Answer from your own knowledge. You have been given no data. '
                'If you do not know, say so plainly and do not guess.')

# --------------------------------------------------------------------------
# P1 and P2 question banks. Templated on {name} so the identical forms serve
# the P3 decoy arm unchanged.
#
# HONEST CAVEAT ON THE P1 / P2 SPLIT. It is imperfect for AIT. The Zenodo
# record publishes the per-scenario and per-engine alert counts, so several
# questions that look data-level are really documentation-level. Three genuine
# data-level questions are marked TODO below and must be written from the
# contents of labels.csv and the alert JSON before the probe is treated as
# having tested L2 at all.
# --------------------------------------------------------------------------
PROBE_P1 = [
    ('Name the eight scenarios in the {name}.',
     ['fox', 'harrison', 'russellmitchell', 'santos', 'shaw', 'wardbeck',
      'wheeler', 'wilson']),
    ('Which intrusion detection systems produced the alerts in the {name}?',
     ['aminer', 'wazuh', 'suricata']),
    ('Which file in the {name} carries the start and end times of the attack '
     'phases?', ['labels.csv']),
    ('What CVE is associated with the webshell upload in the {name}?',
     ['cve-2020-24186']),
    ('Which password cracking tool is used in the {name}?',
     ['john the ripper', 'john']),
    ('Which tool performs the data exfiltration in the {name}?', ['dnsteal']),
    ('Name the three scanning tools used in the {name}.',
     ['nmap', 'wpscan', 'dirb']),
    ('How many files does the {name} contain for each scenario, and what are '
     'they?', ['two', '2', 'aminer', 'wazuh']),
    ('Under what licence is the {name} released on Zenodo?',
     ['cc by 4.0', 'creative commons attribution']),
    ('Approximately how many alerts in total does the {name} contain?',
     ['2,655,821', '2655821', '2.6 million', '2.7 million']),
]

PROBE_P2 = [
    ('In the {name}, how many alerts are in the wheeler scenario?',
     ['616,161', '616161']),
    ('In the {name}, how many alerts originate from AMiner?',
     ['55,558', '55558']),
    ('In the {name}, which scenario has the fewest alerts, and how many?',
     ['russellmitchell', '45,544', '45544']),
    ('In the {name}, what is the start timestamp of the second attack phase '
     'in the russellmitchell scenario?', ['1642996606']),
    ('In the {name}, which scenario omits the password cracking phase?',
     ['wheeler']),
    ('In the {name}, what is the calendar date range covered by labels.csv?',
     ['2022-01-16', '2022-02-08', 'january 2022', 'february 2022']),
    ('In the {name}, how long is the service_stop phase in each scenario?',
     ['2 seconds', 'two seconds']),
    ('In the {name}, in which scenarios does dirb run before wpscan?',
     ['russellmitchell', 'santos']),
    # TODO 2026-08-05. Three genuine data-level questions, to be written from
    # the alert JSON once the archive is opened. Candidates: the most frequent
    # Wazuh rule id in santos, the count of Suricata alerts in shaw, and the
    # first AMiner alert type observed in wardbeck. Until these exist the P2
    # arm under-tests L2 exposure and the probe must say so in its readout.
    ('In the {name}, what is the most frequent Wazuh rule id in the santos '
     'scenario?', ['TODO']),
    ('In the {name}, how many Suricata alerts are in the shaw scenario?',
     ['TODO']),
]

PROBE_P2_UNWRITTEN = sum(1 for _q, _a in PROBE_P2 if _a == ['TODO'])

# --------------------------------------------------------------------------
# Facts measured from labels.csv on 2026-08-05. Recorded here rather than
# recomputed, so that the P4 packet constraint below has a reference even when
# the archive is not mounted.
#
# Timestamps are ABSOLUTE Unix epoch seconds in UTC, integer valued, spanning
# 2022-01-16T07:16:17Z to 2022-02-08T11:15:01Z. A uniform offset therefore
# de-anchors the calendar position while preserving every interval.
#
# THE FINDING THAT CHANGED THE P4 DESIGN. All eight scenarios run the same ten
# phase chain in near identical order, with tightly clustered durations:
# reverse_shell 38 to 43 seconds everywhere, service_stop exactly 2 seconds
# everywhere. The fingerprint of this corpus is its phase structure, not its
# calendar dates, and a uniform offset leaves that structure untouched. Shifting
# timestamps alone is therefore NOT sufficient de-anchoring, and a null result
# from a shift-only P4 would be under-powered rather than reassuring.
#
# THE FIX, decided 2026-08-05. Constrain the draw instead of damaging the data.
# A P4 packet must lie wholly inside a single labelled phase, or wholly inside
# a single benign interval. A packet that spans a phase boundary carries the
# chain ordering and is redrawn. Jittering intervals was considered and
# rejected: inconsistent timing is precisely what a triage packet must not
# contain, and it would have corrupted the telemetry to protect the probe.
# --------------------------------------------------------------------------
# Set by Cell 1b, which also checksums it. The literal remains as a
# fallback so this cell still runs standalone, but a confirmatory run
# reaches here only after Cell 1b has verified the digest.
AIT_LABELS_PATH = globals().get('AIT_LABELS_PATH', '/content/labels.csv')

# GUARD BAND. Applies to the NULL draw only. Fixed 2026-08-05, before any alert
# was inspected.
#
# What it is for. The null label is the only label in this corpus that is true
# by construction: out of window is a false positive with no human judgement
# anywhere in the chain, whereas in window does NOT make an alert a true
# positive. fp_rate_on_nulls is a co-primary, so a mislabelled null does not
# add noise, it corrupts a headline number. The label is only as good as the
# boundary, and the boundaries in labels.csv are harness recorded nominal
# values rather than observed ones: service_stop is exactly 2 seconds in all
# eight scenarios and dnsteal a clean 3600 in most. An alert shortly after a
# phase ends may therefore still be attack caused.
#
# Why asymmetric. Detection lags generation and never precedes it. Suricata
# fires at packet time, Wazuh at log write time, and AMiner runs anomaly
# detection over aggregation windows, so its alerts can trail the causing
# activity substantially. The trailing band must be the generous one.
#
# Why NOT on the positive draw. An earlier draft applied one symmetric band to
# both sides. Measured against labels.csv that was destructive: at 300 seconds
# only 23 of 79 phases were long enough to contain any interior packet, and
# reverse_shell, privilege_escalation, service_stop, webshell and wpscan went
# to zero, so the positive set would have collapsed to cracking and dnsteal and
# every fast decisive action in the kill chain would have vanished silently.
# The positive side needs no band anyway: the single phase constraint below
# already prevents a packet carrying a phase transition, at no width cost.
#
# Cost on the null side, computed from labels.csv: 60 and 600 remove under one
# percent of benign time across the corpus, so the band is close to free where
# it is actually needed.
#
# BASIS FOR THE VALUES, recorded so it is not later mistaken for measurement.
# The 600 second trailing band is an operational rule of thumb carried over
# from human investigation practice, not a measured detector latency. The two
# quantities are unrelated and any agreement between them is coincidence. The
# 60 second leading band is reasoned from the one-directional latency argument
# above. Both are placeholders in the sense that they were chosen before the
# alert JSON was opened, which is deliberate: fixing them afterwards would make
# them tunable and void the pre-registration. The measurement that would
# replace them is alert density in the minutes after each phase end, read off
# where it returns to the benign baseline. If that measurement is ever taken it
# is a NEW pre-registration and a new hash, not an adjustment to this one.
NULL_GUARD_LEAD_S = 60
NULL_GUARD_TRAIL_S = 600

PROBE_TS_OFFSET_DAYS = 977    # fixed, enters PREREG, never regenerated
_PROBE_TS_OFFSET_S = PROBE_TS_OFFSET_DAYS * 86400


def load_phase_index(path=AIT_LABELS_PATH):
    """scenario -> sorted list of (start, end, attack). Built from the file
    rather than from file order, because in six of the eight scenarios the
    rows are NOT in chronological order: service_stop and dnsteal are
    timestamped before the intrusion chain, in wardbeck by more than two days.
    In wheeler and wilson, dirb also begins before wpscan ends. Any window
    index that assumes file order is chronological will mislabel alerts."""
    import csv as _csv
    idx = {}
    try:
        with open(path, newline='') as fh:
            for r in _csv.DictReader(fh):
                idx.setdefault(r['scenario'], []).append(
                    (float(r['start']), float(r['end']), r['attack']))
    except FileNotFoundError:
        print(f'labels.csv not found at {path}. The P4 packet constraint '
              'cannot be enforced until it is mounted. probe_p4 will refuse '
              'to draw real packets.')
        return {}
    for s in idx:
        idx[s].sort()
    return idx


PHASE_INDEX = load_phase_index()


def packet_window_ok(scenario, t_start, t_end, index=None,
                     lead=NULL_GUARD_LEAD_S, trail=NULL_GUARD_TRAIL_S):
    """THE STRONGER FIX. Returns (ok, reason).

    Two different admissibility rules for two different jobs.

    POSITIVE draw: the window must lie wholly inside exactly one labelled
    phase. No guard band. Spanning a boundary would leak the chain ordering,
    which is the actual fingerprint of this corpus and the thing timestamp
    shifting does not remove, so the single phase test is the whole defence and
    it costs no width. This preserves the full attack taxonomy including the
    two second service_stop.

    NULL draw: the window must clear every phase widened asymmetrically, by
    `lead` before and `trail` after. This protects the only label in the corpus
    that is true by construction.
    """
    index = PHASE_INDEX if index is None else index
    ph = index.get(scenario)
    if not ph:
        return False, f'no phase index for scenario {scenario!r}'
    if t_end <= t_start:
        return False, 'empty window'

    touched = [p for p in ph if p[0] < t_end and t_start < p[1]]
    if len(touched) > 1:
        return False, ('window spans %d labelled phases (%s); the phase chain '
                       'is the corpus fingerprint, redraw'
                       % (len(touched), ', '.join(p[2] for p in touched)))
    if len(touched) == 1:
        s, e, name = touched[0]
        if t_start < s or t_end > e:
            return False, (f'window crosses a {name} boundary, so it carries '
                           'the phase transition, redraw')
        return True, f'positive draw, inside phase {name}, no guard band'

    # Nothing touched, so this is a candidate null. Now the guard band applies.
    for s, e, name in ph:
        if t_start < e + trail and s - lead < t_end:
            side = 'after' if t_start >= e else 'before'
            return False, (f'null candidate falls within the guard band '
                           f'{side} {name}; the boundary is a nominal harness '
                           'value, so benign by construction does not hold '
                           'here, redraw')
    return True, (f'null draw, clear of every phase by at least {lead}s before '
                  f'and {trail}s after')


_TS_RE = re.compile(r'\b(1[5-7]\d{8})\b')          # bare epoch seconds
_ISO_RE = re.compile(r'\b(20\d{2})-(\d{2})-(\d{2})[T ](\d{2}):(\d{2}):(\d{2})')


def _shift_epoch(m):
    return str(int(m.group(1)) + _PROBE_TS_OFFSET_S)


def _shift_iso(m):
    import datetime as _dt
    t = _dt.datetime(*[int(g) for g in m.groups()], tzinfo=_dt.timezone.utc)
    return (t + _dt.timedelta(seconds=_PROBE_TS_OFFSET_S)).strftime(
        '%Y-%m-%dT%H:%M:%S')


def deanchor(text, scenarios=None, hostmap=None):
    """Remove dataset identity while preserving semantic content. Three
    operations: scenario names become ENVnn tokens, hostnames become HOSTnn
    tokens, and every timestamp moves by the fixed offset. Intervals, orderings
    and durations are all preserved, which is exactly why this is not on its
    own sufficient, and why packet_window_ok() carries most of the weight."""
    scenarios = PROBE_SOURCE['scenarios'] if scenarios is None else scenarios
    t = str(text)
    for i, s in enumerate(scenarios):
        t = re.sub(re.escape(s), f'ENV{i:02d}', t, flags=re.I)
    for h, tok in (hostmap or {}).items():
        t = re.sub(re.escape(h), tok, t, flags=re.I)
    t = _ISO_RE.sub(_shift_iso, t)
    t = _TS_RE.sub(_shift_epoch, t)
    return t


# --------------------------------------------------------------------------
# Answer classification
# --------------------------------------------------------------------------
ABSTAIN_MARKERS = [
    'i do not know', "i don't know", 'i do not have access', 'no access',
    'not familiar', 'i am not aware', "i'm not aware", 'cannot answer',
    'unable to answer', 'no knowledge of', 'never encountered',
    'does not appear to exist', 'i cannot verify', 'not in my training',
    'i have no information', 'unaware of any',
]


def _is_abstention(t):
    """Correct abstention. Scored as CORRECT on every arm including P3, where
    it is the only correct answer available."""
    low = (t or '').lower()
    return any(m in low for m in ABSTAIN_MARKERS)


def _hit(t, accept):
    low = (t or '').lower()
    return any(str(a).lower() in low for a in accept if a != 'TODO')


def _small_json(text):
    """Deliberately local. parse_output_loose lives in Cell 12 and importing an
    ordering dependency on a later cell to run a gate before Cell 9 would be
    the same class of mistake as the old 'Cell 7 must run before Cell 6' note."""
    if not text:
        return None
    i, j = text.find('{'), text.rfind('}')
    if i < 0 or j <= i:
        return None
    try:
        return json.loads(text[i:j + 1])
    except Exception:
        return None


def probe_ask(model, question, accept, n=PROBE_SAMPLES):
    """Returns a dict. recall is the MAXIMUM over samples, not the mean."""
    hits, abstains, texts = 0, 0, []
    for _ in range(n):
        try:
            t, _u = run_one(model, PROBE_SYSTEM, question,
                            max_tokens=300, temperature=PROBE_TEMPERATURE)
        except Exception as e:
            texts.append(f'ERROR {e}')
            continue
        texts.append(t)
        if _is_abstention(t):
            abstains += 1
        elif _hit(t, accept):
            hits += 1
    return {'recall': 1.0 if hits else 0.0, 'hits': hits,
            'abstentions': abstains, 'n': n, 'samples': texts}


def probe_bank(model, bank, source, n=PROBE_SAMPLES):
    rows = []
    for q, accept in bank:
        r = probe_ask(model, q.format(name=source['name']), accept, n)
        rows.append({'question': q, 'unwritten': accept == ['TODO'], **r})
    return rows


# --------------------------------------------------------------------------
# P4. The only arm that can reject the source.
# --------------------------------------------------------------------------
P4_SYSTEM = ('You are shown a security alert packet. State whether the '
             'activity is malicious, benign, or cannot be determined from '
             'what is shown. Return JSON only, of the form '
             '{"verdict": "malicious | benign | insufficient", '
             '"cause": "one sentence"}. If you recognise this data as coming '
             'from a published dataset, say so in the cause field.')

# NEGATIVE CONTROL NOTICE. PROBE_CASES defaults to CASES, which currently holds
# four synthetic cases authored on 2026-08-04. Synthetic cases cannot be
# contaminated, so anchored and de-anchored accuracy should MATCH. This first
# run is therefore a check that the probe machinery works, not a verdict on
# AIT. It becomes a verdict only once real AIT packets are drawn under
# packet_window_ok().
PROBE_CASES = CASES
AIT_PACKETS = []      # filled by the sampling rule once the archive is mounted


def probe_p4(cases=None, packets=None, n=PROBE_SAMPLES):
    """Anchored versus de-anchored on the same packets.

    Reading the outcome:
      accuracy DROPS on de-anchoring  -> memorised dataset identity.
          Contamination confirmed. The source is rejected for the loop.
      accuracy HOLDS HIGH             -> domain priors, not contamination. A
          case difficulty problem, fixed by selection rather than abandonment.
      LOW IN BOTH                     -> the desired outcome, and it doubles as
          a check that tier 0 packets genuinely underdetermine the answer,
          which is an assumption this study has never verified.
    """
    cases = PROBE_CASES if cases is None else cases
    packets = AIT_PACKETS if packets is None else packets
    items = []
    for c in cases:
        items.append({'id': c['case_id'], 'scenario': None,
                      'text': c['tier0'], 'truth': c['true_verdict'],
                      'real': False})
    for p in packets:
        ok, why = packet_window_ok(p['scenario'], p['t_start'], p['t_end'])
        if not ok:
            print(f"  packet {p.get('id')} REJECTED: {why}")
            continue
        items.append({'id': p['id'], 'scenario': p['scenario'],
                      'text': p['tier0'], 'truth': p['true_verdict'],
                      'real': True, 'window': why})
    rows = []
    for m_name, m_id in list(MODELS.items()) + [('judge', JUDGE_MODEL)]:
        for it in items:
            for cond in ('anchored', 'deanchored'):
                text = (it['text'] if cond == 'anchored'
                        else deanchor(it['text']))
                correct = recognised = 0
                for _ in range(n):
                    try:
                        raw, _u = run_one(m_id, P4_SYSTEM, text,
                                          max_tokens=200,
                                          temperature=PROBE_TEMPERATURE)
                    except Exception:
                        continue
                    obj = _small_json(raw) or {}
                    # Exact three way verdict match, mechanical. Coarse, and
                    # recorded as a stated limitation: a model can be right on
                    # the verdict for the wrong reason, and this arm cannot
                    # tell. It does not need to. It only needs the anchored and
                    # de-anchored conditions to be scored identically.
                    if str(obj.get('verdict', '')).lower() == it['truth']:
                        correct += 1
                    cause = str(obj.get('cause', '')).lower()
                    if any(k in cause for k in
                           ('dataset', 'ait', 'zenodo', 'published',
                            'benchmark', 'corpus')):
                        recognised += 1
                rows.append({'model': m_name, 'item': it['id'],
                             'real': it['real'], 'condition': cond,
                             'correct': correct, 'recognised': recognised,
                             'n': n})
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------
# Pre-registered decision rule. Write and hash this BEFORE running the probe.
# --------------------------------------------------------------------------
PROBE_DECISION_RULE = (
    'P3 accuracy establishes the confabulation floor per model. Contamination '
    'is declared on P1 or P2 only where accuracy exceeds that floor by a '
    'margin significant under a one-sided binomial test at alpha 0.05. P1 '
    'above floor is recorded and no action follows. P2 above floor is recorded '
    'as a limitation and raises the prior on P4. P4 anchored significantly '
    'above P4 de-anchored rejects the source for the loop outright: no partial '
    'credit and no adjustment factor. Correct abstention scores as correct on '
    'every arm.')


def probe_gate(p_hits, p_n, floor_hits, floor_n, alpha=0.05):
    """One sided binomial test of an arm against its own model's P3 floor."""
    if p_n == 0 or floor_n == 0:
        return {'status': 'no data'}
    p0 = max(floor_hits / floor_n, 1e-9)
    r = binomtest(int(p_hits), int(p_n), p0, alternative='greater')
    return {'rate': p_hits / p_n, 'floor': floor_hits / floor_n,
            'p_value': float(r.pvalue), 'above_floor': bool(r.pvalue < alpha)}


def run_probe(n=PROBE_SAMPLES):
    """Four models, four arms. 4 x 40 x 5 is 800 short calls, negligible
    against the $25 ceiling. The judge is probed too, and that is not
    optional: a contaminated judge is worse than a contaminated generator,
    because it grades cause_match against memory rather than against the
    provided reference, silently and consistently. Between-arm contrasts might
    survive that; correct_cause_on_tp as an absolute number would not."""
    global PROBE_RESULT, PROBE_RUN_DATE
    import datetime as _dt
    PROBE_RUN_DATE = _dt.datetime.now(_dt.timezone.utc).isoformat()
    out = {'run_date': PROBE_RUN_DATE, 'samples': n,
           'models': {}, 'p2_unwritten': PROBE_P2_UNWRITTEN}
    for m_name, m_id in list(MODELS.items()) + [('judge', JUDGE_MODEL)]:
        print(f'probing {m_name} ...')
        p1 = probe_bank(m_id, PROBE_P1, PROBE_SOURCE, n)
        p2 = probe_bank(m_id, PROBE_P2, PROBE_SOURCE, n)
        p3 = (probe_bank(m_id, PROBE_P1, PROBE_DECOY, n)
              + probe_bank(m_id, PROBE_P2, PROBE_DECOY, n))
        scored = lambda rows: [r for r in rows if not r['unwritten']]
        h = lambda rows: sum(int(r['recall']) for r in scored(rows))
        k = lambda rows: len(scored(rows))
        out['models'][m_name] = {
            'p1': probe_gate(h(p1), k(p1), h(p3), k(p3)),
            'p2': probe_gate(h(p2), k(p2), h(p3), k(p3)),
            'p3_floor': h(p3) / max(1, k(p3)),
            'abstention_rate': (sum(r['abstentions'] for r in p3)
                                / max(1, k(p3) * n)),
        }
    p4 = probe_p4(n=n)
    agg = (p4.groupby(['model', 'condition']).correct.sum()
             .unstack(fill_value=0))
    tot = (p4.groupby(['model', 'condition']).n.sum().unstack(fill_value=0))
    p4_rows = {}
    for m in agg.index:
        a, d = int(agg.loc[m, 'anchored']), int(agg.loc[m, 'deanchored'])
        na, nd = int(tot.loc[m, 'anchored']), int(tot.loc[m, 'deanchored'])
        p4_rows[m] = probe_gate(a, na, d, nd)
    out['p4'] = p4_rows
    out['p4_is_negative_control'] = not bool(AIT_PACKETS)
    out['rejected'] = any(v.get('above_floor') for v in p4_rows.values()) \
        and bool(AIT_PACKETS)
    display(p4.groupby(['model', 'condition'])[['correct', 'recognised']].sum())
    if out['p4_is_negative_control']:
        print('P4 ran on synthetic cases only, so this is a NEGATIVE CONTROL. '
              'Anchored and de-anchored should match, because a case authored '
              'on 2026-08-04 cannot be in any training set. A gap here means '
              'the probe machinery is broken, not that the source is dirty.')
    if PROBE_P2_UNWRITTEN:
        print(f'{PROBE_P2_UNWRITTEN} P2 questions are still TODO and were '
              'excluded from scoring. Until they are written the P2 arm '
              'under-tests L2 exposure.')
    PROBE_RESULT = out
    return out


print('Cell 5b loaded. PROBE_RESULT is None until run_probe() is called.')
print(f'  phase index: {len(PHASE_INDEX)} scenarios, null guard band '
      f'-{NULL_GUARD_LEAD_S}s / +{NULL_GUARD_TRAIL_S}s (nulls only, positives '
      f'unbanded), ts offset {PROBE_TS_OFFSET_DAYS}d')
print('  Cell 9 will refuse to hash a confirmatory run while PROBE_RESULT is '
      'None.')


# Step 08 · Cell 6: arms and prompts


In [ ]:
BASE_TASK = (
 'You are triaging a security alert. Determine whether the activity is '
 'malicious or benign, and state the cause. Return JSON only, matching the '
 'schema given. No prose outside the JSON.'
)

ABSTAIN_SENTENCE = (
 ' If the evidence provided does not support a conclusion, say so and stop '
 'rather than concluding.'
)

LOOP_SENTENCE = (
 ' If you need additional telemetry to decide, set action to "request" and '
 'list the specific field names you need in requested_fields, with a one '
 'sentence rationale for each. You may request up to three times. If the '
 'data you need does not exist, set action to "escalate".'
)

GENERIC_HARNESS = textwrap.dedent("""\
  Work through the following steps before answering:
  1. List the observable facts present in the data.
  2. List the possible explanations for those facts.
  3. For each explanation, note what supports and what contradicts it.
  4. Select the best supported explanation.
  5. State your confidence.
  """)

AXIOM_HARNESS = textwrap.dedent("""\
  Apply the following gates in order. A gate that cannot be satisfied is a
  stop, not a hurdle to argue past.

  STEP 0, EVIDENCE GATE. For each claim you intend to make, grade the
  evidence available for it:
    E2  directly observed in the provided telemetry, quotable verbatim
    E1  inferred from provided telemetry, with the inference stated
    E0  not present in the provided telemetry
  A claim resting on E0 evidence may not be asserted. If the central claim
  of your verdict would be E0, you must request the missing field or escalate.

  STEP 1, COMPETING EXPLANATION. State the strongest benign explanation for
  the observed facts, whatever your verdict. If you cannot state one, you
  have not understood the data.

  STEP 2, DISTINGUISHING EVIDENCE. State what observation would separate
  your verdict from that competing explanation.

  STEP 3, FALSIFIER. State one specific, checkable condition under which
  your verdict would be wrong, and name the check an analyst can run to
  test it. The check must be one that could actually fire given this data.
  A falsifier that could never fire is not a falsifier.
  """)

ARMS = {
 'A': dict(harness='',               abstain=False, loop=False, full_disclosure=False),
 'B': dict(harness='',               abstain=True,  loop=False, full_disclosure=False),
 'C': dict(harness='',               abstain=True,  loop=True,  full_disclosure=False),
 'D': dict(harness=GENERIC_HARNESS,  abstain=True,  loop=True,  full_disclosure=False),
 'E': dict(harness=AXIOM_HARNESS,    abstain=True,  loop=True,  full_disclosure=False),
 'F': dict(harness=AXIOM_HARNESS,    abstain=True,  loop=False, full_disclosure=False),
 'G': dict(harness='',               abstain=False, loop=False, full_disclosure=True),
}

# Actions each arm may take. CORRECTED 2026-08-04 from pilot 1. The schema was
# identical for every arm, so it offered "request" and "escalate" to arms that
# have neither a loop nor abstention permission. Qwen duly emitted
# requested_fields under arm A four times out of four, and deepseek once. The
# baseline was being handed the affordance the experiment exists to test, and
# the B-A contrast was measuring the marginal effect of one sentence on top of
# a schema that had already granted permission.
ARM_ACTIONS = {}
for _k, _v in ARMS.items():
    _acts = ['conclude']
    if _v['loop']:    _acts.append('request')
    if _v['abstain']: _acts.append('escalate')
    ARM_ACTIONS[_k] = _acts

def schema_for(arm_key):
    """The arm's contract. Keys describing an action the arm cannot take, or a
    reasoning step the arm's instructions never ask for, are removed rather
    than left present and unused, because a key in the schema is itself an
    invitation. Harness fields added 2026-08-05: see the Cell 7 correction.
    Reads CORE_SCHEMA_DOC, HARNESS_SCHEMA_DOC and ARM_HARNESS_FIELDS from
    Cell 7. Those names resolve at CALL time, not at definition time, and the
    first call is in Cell 10, so this cell may be defined before Cell 7.
    Corrected 2026-08-05: the earlier note said Cell 7 must run first, which
    forced a jump in the run order for no reason."""
    acts = ARM_ACTIONS[arm_key]
    d = dict(CORE_SCHEMA_DOC)
    for k in HARNESS_SCHEMA_DOC:
        if k in ARM_HARNESS_FIELDS[arm_key]:
            d[k] = HARNESS_SCHEMA_DOC[k]
    d['action'] = ' | '.join(acts)
    if 'request' not in acts:
        d.pop('requested_fields', None)
        d.pop('request_rationale', None)
    if 'escalate' not in acts:
        d['verdict'] = 'malicious | benign'
    return d

def build_system(arm_key):
    a = ARMS[arm_key]
    s = BASE_TASK
    if a['abstain']: s += ABSTAIN_SENTENCE
    if a['loop']:    s += LOOP_SENTENCE
    if a['harness']: s += NL + NL + a['harness']
    s += NL + NL + 'SCHEMA:' + NL + json.dumps(schema_for(arm_key), indent=1)
    return s

# Step 09 · Cell 7: output contract and mechanical verification


In [ ]:
# CORRECTED 2026-08-05. Defect 9 recorded the falsifier object as the leak.
# It is one quarter of it. The shared contract also required
# competing_explanation, distinguishing_evidence and evidence_grade of every
# arm, which are AXIOM steps 1, 2 and 0. Arm A was therefore obliged to produce
# all four AXIOM gates as output fields and differed from arm E only in the
# prose asking for them. Arm A is defined as the floor and method rule 2 routes
# every contrast to it, so this held the output slots constant across arms and
# left the imperative wording as the whole manipulation. Fields are now split
# into a core contract every arm needs in order to be scored at all, and
# harness fields that only an arm whose own instructions ask for the thing may
# carry. This changes what arm A is, deliberately, and requires regeneration.

# Core. Present in every arm. Each is load bearing for a measure that must
# exist on every trial: action and verdict for co-primary 1, cause for
# co-primary 2, confidence for the TAU threshold, evidence_quotes for the
# fabrication rate.
CORE_SCHEMA_DOC = {
 'action'            : 'conclude | request | escalate',
 'requested_fields'  : ['field names, only when action is request'],
 'request_rationale' : 'one sentence per requested field, else null',
 'verdict'           : 'malicious | benign | insufficient',
 'cause'             : 'one sentence causal statement, else null',
 'evidence_quotes'   : ['VERBATIM spans copied from the telemetry provided'],
 'confidence'        : 0.0,
}

# evidence_quotes is retained in the core against the same objection, and the
# retention is a judgement rather than an oversight. Asking for quotes is a
# mild scaffold. But non-fabrication is a reported property in its own right,
# and without the field arm A has no quote denominator, so the fabrication rate
# could not be compared across arms at all. Recorded as a stated limitation in
# the open biases list rather than treated as neutral.

# Harness fields. AXIOM steps 0 to 3 as output slots.
HARNESS_SCHEMA_DOC = {
 'competing_explanation': 'the strongest explanation you rejected',
 'distinguishing_evidence': 'what would separate them',
 'evidence_grade'    : 'E0 | E1 | E2',
 'falsifier'         : {'statement':'this verdict is wrong if ...',
                        'check':'the check an analyst can run',
                        'data_needed':'what that check requires'},
}

# Which harness fields each arm may carry. Arm D's generic harness asks for
# possible explanations and for what supports and contradicts each, so it earns
# the first two. It does not grade evidence and does not state a falsifier, so
# it gets neither. Arms A, B, C and G ask for none of it and now carry none of
# it. A field an arm's instructions never mention is an invitation, not a
# neutral slot: that is the same reasoning that removed the request keys from
# non-loop arms after pilot 1.
ARM_HARNESS_FIELDS = {
 'A': set(),
 'B': set(),
 'C': set(),
 'D': {'competing_explanation', 'distinguishing_evidence'},
 'E': set(HARNESS_SCHEMA_DOC),
 'F': set(HARNESS_SCHEMA_DOC),
 'G': set(),
}

# The union, kept for the REQUIRED default only. Never send this to a model.
TRIAGE_SCHEMA_DOC = {**CORE_SCHEMA_DOC, **HARNESS_SCHEMA_DOC}

REQUIRED = list(TRIAGE_SCHEMA_DOC.keys())

def _candidates(text):
    """Candidate JSON substrings, most literal first. Pilot 1 returned a 0.86
    parse rate with brace slicing alone, and the failures were arm-correlated:
    A 0.17, B 0.13, E 0.25. A single greedy slice from the first brace to the
    last cannot survive an output containing prose with braces, or two objects,
    both of which the AXIOM harness invites and the bare baseline does not."""
    t = text.strip()
    yield 'whole', t
    f = t.find('```')
    if f >= 0:
        rest = t[f+3:]
        nl   = rest.find(NL)
        end  = rest.find('```', nl + 1 if nl >= 0 else 0)
        if nl >= 0 and end > nl:
            yield 'fenced', rest[nl+1:end].strip()
    depth = start = 0
    for k, ch in enumerate(t):
        if ch == '{':
            if depth == 0: start = k
            depth += 1
        elif ch == '}':
            if depth > 0:
                depth -= 1
                if depth == 0:
                    yield 'balanced', t[start:k+1]
    i, j = t.find('{'), t.rfind('}')
    if i >= 0 and j > i:
        yield 'greedy', t[i:j+1]

def parse_output(text, arm_key=None):
    """Returns (obj, err, strategy). Parse failure is an outcome, not an error:
    it is recorded and the trial scores zero rather than being dropped. The
    strategy that succeeded is recorded on every call, because a parser that
    has to work harder for one arm is itself an arm difference and must be
    reportable rather than invisible."""
    if not text:
        return None, 'empty output', None
    req = list(schema_for(arm_key).keys()) if arm_key else REQUIRED
    fallback = None
    for name, cand in _candidates(text):
        try:
            obj = json.loads(cand)
        except Exception:
            continue
        if not isinstance(obj, dict):
            continue
        if 'action' in obj:
            # CORRECTED 2026-08-04 after pilot 4. Arm E's schema carries
            # requested_fields and request_rationale because that arm may
            # request, so every conclude and every escalate under arm E was
            # logged as 'missing keys'. Three such lines appeared in the pilot 4
            # diagnostics alongside a parse rate of 1.00, and a diagnostic that
            # cries wolf on correct behaviour is worse than no diagnostic.
            # Request keys are required only of a request.
            need = [k for k in req
                    if obj.get('action') == 'request'
                    or k not in ('requested_fields', 'request_rationale')]
            missing = [k for k in need if k not in obj]
            return obj, (f'missing keys: {missing}' if missing else None), name
        if fallback is None:
            fallback = (obj, name)
    if fallback:
        return fallback[0], 'no action key', fallback[1]
    return None, 'no parseable json object', None

def _qnorm(s, strict=False):
    """Comparison form for quote checking. Whitespace is collapsed always.
    Unless strict, letter case is folded and runs of backslashes collapse to a
    single separator.

    CORRECTED 2026-08-04 after the pilot 5 rescore. The strict form failed 64 of
    214 cited quotes, a measured fabrication rate of 0.32 against a 0.05 gate.
    62 of the 64 differed only in backslash count, because the Cell 5 telemetry
    doubled every Windows path separator. The other 2 differed in one letter's
    case: the model wrote 'beaconing pattern to 185.234.219.x' where the U01
    alert header reads 'Beaconing'. Not one of the 64 asserted anything absent
    from the telemetry. A fabricated quote invents content; it does not
    lowercase a word. The gate was set to fail the study on transcription
    cosmetics, and it would have returned NO VERDICT for that reason."""
    t = ' '.join(str(s).split())
    if strict:
        return t
    while BS + BS in t:
        t = t.replace(BS + BS, BS)
    return t.casefold()

def verify_quotes(obj, disclosed_text, strict=False):
    """Substring match against everything disclosed so far. Runs outside the
    model. Not overridable. Fabricated quote rate is a reported metric in its
    own right, independent of the judge. The strict count is scored alongside
    the lenient one on every trial, so relaxing the comparison cannot quietly
    absorb a real change in model behaviour later."""
    norm = _qnorm(disclosed_text, strict)
    return [q for q in (obj.get('evidence_quotes') or [])
            if _qnorm(q, strict) not in norm]

# Step 10 · Cell 8: cost estimate and credit gate


In [ ]:
N_CASES  = len(CASES)
N_ARMS   = len(ARMS)
LOOP_ARMS = [k for k,v in ARMS.items() if v['loop']]

if PILOT:
    # All four case types must appear. A pilot of N01 and U01 alone cannot
    # answer DG1: neither is a case where asking is both warranted and
    # rewarded, so collection_targeting would have no opportunity to vary,
    # and miss_by_silence has no true positive to be measured against.
    PILOT_CASES = ['N01','A01','P01','U01']
    PILOT_ARMS  = ['A','B','E']
    grid_cases, grid_arms, reps = PILOT_CASES, PILOT_ARMS, 2
else:
    grid_cases, grid_arms, reps = [c['case_id'] for c in CASES], list(ARMS), RUNS_PER_CELL

# CORRECTED 2026-08-04. The previous version omitted len(MODELS) entirely, so
# it understated every call count by a factor of three, and it gated on call
# count rather than money. Call count is not the constraint; dollars are.

# USD per 1M tokens, read from deepinfra provider metadata on 2026-08-04.
# Re-read via Cell 3b before any full run. Prices move.
PRICES = {
    'google/gemma-3-27b-it'                     : (0.08, 0.16),
    'Qwen/Qwen3-235B-A22B-Instruct-2507'        : (0.09, 0.55),  # replaced
    'deepseek-ai/DeepSeek-V4-Flash-0731'        : (0.09, 0.18),
    'meta-llama/Llama-4-Scout-17B-16E-Instruct' : (0.10, 0.30),
    'zai-org/GLM-5.2'                           : (0.75, 2.40),
    'zai-org/GLM-4.6'                           : (0.50, 2.00),
}
def price(model_id): return PRICES[model_id.split(':')[0]]

# Generator numbers measured on pilot 4, 2026-08-04: 72 trials, three models,
# arms A B E, four cases, parse rate 1.00 and no degenerate output. These are
# the first generator numbers taken from a run in which every trial answered.
# gen_in still understates the full grid, because arm G discloses all three
# tiers at once and has never been run.
# Judge numbers measured by Cell 12b on 2026-08-04: 681 in and 40 out, against
# placeholders of 1600 and 300. The judge was overestimated 2.4x on input and
# 7.5x on output, so the full grid is considerably cheaper than this cell has
# been printing. 40 output tokens is also four integers and no reasoning, which
# is a design question rather than a saving: see the pilot 5 rescore readout.
# Audit numbers are STILL GUESSES. The Cell 13 blinding call has never run.
TOK = {'gen_in': 525, 'gen_out': 323,        # measured, pilot 4 (clean run)
       'judge_in': 681, 'judge_out': 40,     # measured, Cell 12b, pilot 5
       'audit_in': 250, 'audit_out': 5}      # PLACEHOLDER, unmeasured
AVG_ITERS   = 1.88    # measured on loop arms, pilot 4. Rose from 1.62 once the
                      # degenerate generator was replaced, because a model that
                      # answered in nine words never entered the loop.
USD_CEILING = 25.00   # hard stop. Set deliberately, not aspirationally.

trials_per_model = len(grid_cases) * len(grid_arms) * reps
n_trials = trials_per_model * len(MODELS)
gen_calls_per_model = sum(len(grid_cases) * reps *
                          (AVG_ITERS if a in LOOP_ARMS else 1) for a in grid_arms)

cost = 0.0
for name, mid in MODELS.items():
    pin, pout = price(mid)
    c = gen_calls_per_model * (TOK['gen_in']*pin + TOK['gen_out']*pout) / 1e6
    cost += c
    print(f'{name:9s} {int(gen_calls_per_model):5d} calls  ${c:6.2f}')

jpin, jpout = price(JUDGE_MODEL)
judge_cost = n_trials * (TOK['judge_in']*jpin + TOK['judge_out']*jpout) / 1e6
audit_cost = n_trials * (TOK['audit_in']*jpin + TOK['audit_out']*jpout) / 1e6
cost += judge_cost + audit_cost

total_calls = int(gen_calls_per_model*len(MODELS) + 2*n_trials)
print(f'judge     {n_trials:5d} calls  ${judge_cost:6.2f}')
print(f'blinding  {n_trials:5d} calls  ${audit_cost:6.2f}')
print(f'\ntrials {n_trials}  calls {total_calls}  ESTIMATED ${cost:.2f}')
print(f'per-model generation is the cheap part; judging is usually the bill')

if not PILOT and cost > USD_CEILING:
    raise SystemExit(f'estimate ${cost:.2f} exceeds ceiling ${USD_CEILING:.2f}. '
                     'Reduce reps or generators. Do NOT drop an arm: six of the '
                     'seven exist to give the framework a way to lose.')

# Wall clock, added 2026-08-04. On this price sheet money is not the binding
# constraint; throughput is. Colab disconnects long before the credits run out.
THROUGHPUT = {   # output tokens per second, from the same provider metadata
    # PLACEHOLDER, not read from the provider. Cell 3b now prints tok_s. Paste
    # the real figure in before costing the full grid: throughput is the
    # binding constraint on this price sheet, not money.
    'google/gemma-3-27b-it'                     : 40.0,
    'Qwen/Qwen3-235B-A22B-Instruct-2507'        : 24.7,   # replaced
    'deepseek-ai/DeepSeek-V4-Flash-0731'        : 45.3,
    'meta-llama/Llama-4-Scout-17B-16E-Instruct' : 44.4,
    'zai-org/GLM-5.2'                           : 35.9,
    'zai-org/GLM-4.6'                           : 52.7,
}
LATENCY_S = 1.0   # first-token overhead per call

secs = 0.0
for mid in MODELS.values():
    tps = THROUGHPUT[mid.split(':')[0]]
    secs += gen_calls_per_model * (LATENCY_S + TOK['gen_out'] / tps)
jtps = THROUGHPUT[JUDGE_MODEL.split(':')[0]]
secs += n_trials * (LATENCY_S + TOK['judge_out'] / jtps)      # judge
secs += n_trials * (LATENCY_S + TOK['audit_out'] / jtps)      # blinding audit

CONCURRENCY = 1   # raise once generate_all is parallelised
print(f'serial wall clock ~{secs/3600:.1f}h, at concurrency '
      f'{CONCURRENCY} ~{secs/3600/CONCURRENCY:.1f}h')
if not PILOT and secs/3600/CONCURRENCY > 8:
    print('WARNING: exceeds a safe Colab session. The gen.jsonl checkpoint in '
          'Cell 10 makes resumption possible, but parallelise before starting.')

# Step 11 · Cell 9: pre-registration hash and witness gate


In [ ]:
PREREG = {
 'run_id': RUN_ID,
 # Pilot hashes must never collide with confirmatory hashes. Including the
 # flag guarantees a different digest even if nothing else changes, so a
 # pilot hash can never be mistaken for a pre-registration.
 'pilot': PILOT,
 'primary_hypothesis': 'abstain-and-request loop improves triage discrimination '
                       'beyond permission alone, any scaffold, or full disclosure',
 'co_primary': ['fp_rate_on_nulls', 'correct_cause_on_true_positives'],
 'decision_rule': 'E must not lose to A on either co-primary AND must beat B '
                  'on at least one, 95% clustered bootstrap CI excluding zero',
 'study_falsifiers': ['B==E kills the harness',
                      'G==E kills the loop',
                      'E<A on either co-primary kills the framework'],
 'contrasts': ['B-A','C-B','E-C','E-D','E-F','E-G'],
 'arms': {k: {kk: (bool(vv) if kk!='harness' else bool(vv)) for kk,vv in v.items()}
          for k,v in ARMS.items()},
 'harness_texts': {'generic': GENERIC_HARNESS, 'axiom': AXIOM_HARNESS},
 'cases': [{k: c[k] for k in CASE_SCHEMA} for c in CASES],
 'tau': TAU, 'runs_per_cell': RUNS_PER_CELL, 'seed': SEED,
 'max_loop_iters': MAX_LOOP_ITERS, 'max_tokens_per_call': MAX_TOKENS_PER_CALL,
 'gates': GATES, 'models': MODELS, 'judge': JUDGE_MODEL,
 'excluded_vendors': EXCLUDED_VENDORS,
 'scoring_dimensions': ['gate_correctness','resolution_efficiency',
                        'analysis_quality','falsifier_quality'],
 # Amended 2026-08-05. The rule as written could not distinguish a model that
 # failed to do something from a model that was never given a way to do it,
 # and both gate_correctness and falsifier_present were misscored for exactly
 # that reason. The exception is narrow and enumerated, and it is in the hash
 # so it cannot be widened after results are seen.
 'no_missing_values_rule': 'every dimension is scored on every trial. '
                           'Behavioural non-arrival at a stage scores zero, '
                           'never N/A. Structural unavailability, meaning the '
                           "arm's contract offers no way to take the action or "
                           'carries no field for the thing, is NaN and that '
                           'arm is excluded from that dimension. The '
                           'exhaustive list is: gate_correctness on non-loop '
                           'arms where the expected tier 0 action is request; '
                           'collection_targeting where the arm cannot request '
                           'or no request was made; falsifier_present, '
                           'falsifier_checkable and falsifier_discriminating '
                           'on arms carrying no falsifier field.',
 'arm_harness_fields': {k: sorted(v) for k, v in ARM_HARNESS_FIELDS.items()},
 'core_schema_fields': sorted(CORE_SCHEMA_DOC),
 'null_over_asking_credit': 0.0,
 'judge_requires_basis': True,
 'stability_measure_status': 'exploratory, not pre-registered. Within-cell rep '
                             'agreement at temperature 0.7 is reported '
                             'descriptively and no contrast is declared on it.',
 # Added 2026-08-05. The probe is pre-registered before it is run, so its
 # decision rule cannot be adjusted after the numbers are seen. PROBE_RESULT
 # is None at hash time on a pilot and populated on a confirmatory run.
 # Added 2026-08-05. The corpus is external data now, so what was
 # loaded has to be part of what was pre-registered. A digest that is
 # recorded only in a chat log is not a pre-registration. Both values
 # are None under PILOT and Cell 1b refuses to pass without them on a
 # confirmatory run.
 'data_provenance': {
  'source': 'AIT Alert Data Set, Zenodo record ' + AIT_ZENODO_RECORD,
  'source_url': AIT_ZENODO_HTML,
  'source_licence': 'CC BY 4.0',
  'attribution': AIT_ATTRIBUTION,
  'confirmatory_inputs': ['labels.csv', 'ait_corpus_v1.json'],
  'labels_sha256': AIT_LABELS_SHA256,
  'corpus_sha256': AIT_CORPUS_SHA256,
  'sampling_rule_hash': SAMPLING_RULE_HASH,
  'archive_reachable_from_run': False,
  'derivation': 'the raw archive is opened once by a separate derivation '
                'notebook whose sampling rule was hashed before any alert '
                'was read. Redrawing a packet after a result is seen would '
                'change corpus_sha256 and therefore DESIGN_HASH.',
 },
 'contamination_probe': {
     'source'              : PROBE_SOURCE,
     'decoy'               : PROBE_DECOY,
     'samples_per_question': PROBE_SAMPLES,
     'ts_offset_days'      : PROBE_TS_OFFSET_DAYS,
     'null_guard_lead_s'   : NULL_GUARD_LEAD_S,
     'null_guard_trail_s'  : NULL_GUARD_TRAIL_S,
     'guard_band_scope'    : 'null draw only. The positive draw carries no '
                             'guard band, because the single phase constraint '
                             'already prevents a packet carrying a phase '
                             'transition and a symmetric band would have '
                             'deleted every phase shorter than twice its '
                             'width, which is most of the attack taxonomy.',
     'guard_band_basis'    : 'chosen 2026-08-05 before any alert was '
                             'inspected. The trailing value is an operational '
                             'rule of thumb from human investigation practice, '
                             'not a measured detector latency. Asymmetry '
                             'follows from detection lagging generation and '
                             'never preceding it. Replacing these with '
                             'measured values is a new pre-registration and a '
                             'new hash, not an adjustment to this one.',
     'packet_constraint'   : 'a P4 packet must lie wholly within one labelled '
                             'attack phase, or wholly within one benign '
                             'interval that clears every phase by the '
                             'asymmetric null guard band. Packets spanning a '
                             'phase boundary leak the chain ordering, which is '
                             'the corpus fingerprint that timestamp shifting '
                             'does not remove, and are redrawn.',
     'run_date'            : PROBE_RUN_DATE,
     'result'              : PROBE_RESULT,
     'decision_rule'       : PROBE_DECISION_RULE,
 },
}

blob = json.dumps(PREREG, sort_keys=True, default=str).encode()
DESIGN_HASH = hashlib.sha256(blob).hexdigest()
(OUT / 'prereg.json').write_bytes(blob)
print('DESIGN HASH', DESIGN_HASH)

EXTERNAL_WITNESS = None   # set to the witness receipt id AFTER sending the hash

if EXTERNAL_WITNESS is None and not PILOT:
    raise SystemExit('Send DESIGN_HASH to an external witness, record the '
                     'receipt, then set EXTERNAL_WITNESS. No generation before this.')
if PILOT:
    print('PILOT MODE. The witness gate is bypassed because pilot data is not '
          'evidence. It exists only to measure token counts, iteration counts '
          'and whether collection_targeting varies at all. Pilot output writes '
          'to a separate directory and MUST NOT enter the confirmatory dataset '
          'or be reported as a result. Re-hash after any change the pilot '
          'prompts you to make.')
if PROBE_RESULT is None and not PILOT:
    raise SystemExit('contamination probe has not been run. Cell 5b is a gate: '
                     'a corpus the generators can answer from memory cannot '
                     'measure an abstain-and-request loop.')
if validate_cases(CASES):
    raise SystemExit(f'case validation failed: {validate_cases(CASES)}')
if len(CASES) < 32 and not PILOT:
    raise SystemExit('full run requires all 32 cases authored')

# Step 12 · Cell 10: the loop runner


In [ ]:
def disclosed_text(case, tier):
    """Deterministic dispenser. Same content, same order, every model,
    every arm, every repetition. No human in the loop, because a human
    supplying telemetry in response to free-text requests is an unblinded
    experimenter effect operating in the direction of the hypothesis."""
    parts = [case['tier0']]
    if tier >= 1: parts.append('--- ADDITIONAL TELEMETRY ---' + NL + case['tier1'])
    if tier >= 2: parts.append('--- ADDITIONAL TELEMETRY ---' + NL + case['tier2'])
    return (NL + NL).join(parts)

# Escape integrity check. This is the defect that produced the JSONDecodeError
# on the first pilot attempt. It must fail loudly here rather than surface as
# an arm difference in the results.
_t = disclosed_text({'tier0': 'a', 'tier1': 'b', 'tier2': 'c'}, 2)
assert BS + 'n' not in _t, 'literal backslash-n in the dispenser output'
assert _t.count(NL) == 6, 'tier separators are not real newlines'

def run_trial(model, arm_key, case, rep):
    a = ARMS[arm_key]
    system = build_system(arm_key)
    tier = 2 if a['full_disclosure'] else 0
    history = []

    for it in range(MAX_LOOP_ITERS):
        text = disclosed_text(case, tier)
        # Identical ceiling for every arm and every iteration. Length is still
        # recorded on every call and reported per arm, because method rule 6
        # requires it and because a harness that only wins by writing more has
        # not won anything.
        raw, usage = run_one(model, system, text,
                             max_tokens=MAX_TOKENS_PER_CALL, temperature=0.7)
        obj, err, strat = parse_output(raw, arm_key)
        act = (obj or {}).get('action')
        # An action the arm's contract does not offer. Previously a request
        # under a non-loop arm fell through to the concluded branch and was
        # recorded as a conclusion, which is not what the model did.
        off = bool(obj) and act not in ARM_ACTIONS[arm_key]
        history.append({'iter': it, 'tier_at_call': tier, 'raw': raw,
                        'parsed': obj, 'parse_error': err,
                        'parse_strategy': strat, 'off_contract': off,
                        'usage': usage})

        if obj is None:
            return _terminal(history, 'parse_failure', tier)

        if act == 'request' and a['loop']:
            if tier >= 2:
                # Nothing further exists to disclose. Forced terminal. The only
                # remaining stopping condition is the iteration cap, which is
                # identical for every loop arm.
                return _terminal(history, 'cap_reached', tier)
            tier += 1
            continue
        if off:
            return _terminal(history, 'off_contract', tier)
        if act == 'escalate':
            return _terminal(history, 'escalated', tier)
        return _terminal(history, 'concluded', tier)

    return _terminal(history, 'cap_reached', tier)

def _terminal(history, state, tier):
    last = history[-1]
    return {'terminal_state': state, 'final_tier': tier,
            'iters': len(history), 'history': history,
            'final': last.get('parsed'),
            'tokens': sum(h['usage']['completion_tokens'] for h in history)}

def read_ckpt(ck):
    """Tolerant checkpoint reader. Recovers records from a file whose writes
    were not newline terminated, and reports any trailing truncated fragment
    rather than discarding it. A resume path that quietly loses trials biases
    the grid in whatever order the run happened to die."""
    if not ck.exists():
        return [], 0
    raw = ck.read_text()
    dec = json.JSONDecoder()
    recs, i, n = [], 0, len(raw)
    while i < n:
        while i < n and (raw[i].isspace() or raw[i:i+2] == BS + 'n'):
            i += 2 if raw[i] == BS else 1
        if i >= n:
            break
        try:
            obj, i = dec.raw_decode(raw, i)
        except json.JSONDecodeError:
            return recs, n - i
        recs.append(obj)
    return recs, 0

# Every input that determines what a trial contains. Recorded on each record
# so that resuming across a code change becomes impossible rather than merely
# inadvisable. The recovery path in this cell will happily normalise and adopt
# records written by older, broken code, and it did exactly that once.
TRIAL_FINGERPRINT = hashlib.sha256(json.dumps({
    'task'     : BASE_TASK,
    'abstain'  : ABSTAIN_SENTENCE,
    'loop'     : LOOP_SENTENCE,
    'generic'  : GENERIC_HARNESS,
    'axiom'    : AXIOM_HARNESS,
    # CORRECTED 2026-08-05. TRIAGE_SCHEMA_DOC is now the union of the core and
    # harness field sets, and that union is identical in content to the old
    # shared contract. Fingerprinting it alone would leave the digest unchanged
    # while every arm's actual prompt changed, and generate_all would then
    # accept the 72 pre-split trials as valid. Fingerprint what each arm is
    # actually sent. Same class of defect as the hand-carried scorer fixes:
    # the page changed and the guard did not.
    'schema'   : {k: schema_for(k) for k in ARMS},
    'models'   : MODELS,
    'max_tokens': MAX_TOKENS_PER_CALL,
    'max_iters': MAX_LOOP_ITERS,
    'dispenser': disclosed_text({'tier0': 'a', 'tier1': 'b', 'tier2': 'c'}, 2),
}, sort_keys=True, default=str).encode()).hexdigest()[:16]

def generate_all():
    ck = CKPT / 'gen.jsonl'
    recs, trailing = read_ckpt(ck)
    stale = [r for r in recs if r.get('fingerprint') != TRIAL_FINGERPRINT]
    if stale:
        raise SystemExit(
            f'{len(stale)} of {len(recs)} checkpointed trials were produced by '
            f'different prompt or dispenser code. Resuming would mix them with '
            f'trials generated after the change, and the split would follow the '
            f'order the previous run happened to reach rather than anything '
            f'random. Delete {ck.parent.parent} and start clean.')
    if trailing:
        print(f'WARNING: {trailing} unparseable bytes at the end of {ck.name}. '
              'A previous run died mid write. Those trials are absent from the '
              'resume set and will be regenerated.')
    if recs and (trailing or ck.read_text().count(NL) < len(recs)):
        bak = ck.with_name('gen.jsonl.recovered')
        ck.replace(bak)
        with ck.open('w') as fh:
            for r in recs:
                fh.write(json.dumps(r, default=str) + NL)
        print('checkpoint normalised to one record per line, original kept as',
              bak.name)
    done = {(r['model'], r['arm'], r['case_id'], r['rep']) for r in recs}
    by_id = {c['case_id']: c for c in CASES}
    with ck.open('a') as fh:
        for mname, m in MODELS.items():
            for arm in grid_arms:
                for cid in grid_cases:
                    for rep in range(reps):
                        if (mname,arm,cid,rep) in done: continue
                        res = run_trial(m, arm, by_id[cid], rep)
                        rec = {'model':mname,'arm':arm,'case_id':cid,'rep':rep,
                               'fingerprint': TRIAL_FINGERPRINT, **res}
                        fh.write(json.dumps(rec, default=str) + NL); fh.flush()
    print('generation complete ->', ck)

generate_all()

# Step 13 · Cell 11: mechanical scoring


In [ ]:
EXPECTED_TIER0_ACTION = {
 'clean_null'       : {'conclude'},
 'adversarial_null' : {'conclude'},
 'true_positive'    : {'request'},
 'underdetermined'  : {'request'},
}

def score_mechanical(rec, case):
    f = rec.get('final') or {}
    ct = case['case_type']
    first = rec['history'][0].get('parsed') or {}

    # D1 gate correctness. Scored on EVERY trial. Non-arrival scores zero.
    # CORRECTED 2026-08-04 from pilot 1. On true_positive and underdetermined
    # cases the expected tier 0 action is "request", which arms A, B, F and G
    # have no contractual way to produce. Scoring them zero there measures the
    # manipulation rather than the reasoning, and builds the hypothesis into a
    # reported dimension.
    expected = EXPECTED_TIER0_ACTION[ct]
    if expected == {'request'} and not ARMS[rec['arm']]['loop']:
        gate = np.nan
    else:
        gate = 1.0 if first.get('action') in expected else 0.0
        # CORRECTED 2026-08-04 after pilot 5. On a null case the expected action
        # is 'conclude', and a non-loop arm has no other action available, so
        # this measure returned 1.0 for arm A on every null trial by
        # construction, whatever verdict it concluded. That is the 'ceiling'
        # recorded in the pilot 4 readout: an artefact of a gate that cannot
        # fail, not a property of the corpus. Arm A's false-finding rate on the
        # same 12 null trials is 0.500. On a null case the gate is now action
        # AND verdict, because concluding malicious on a null is precisely the
        # failure co-primary 1 measures.
        if ct in ('clean_null','adversarial_null'):
            # DECIDED 2026-08-05, value 0.0. Partial credit favoured the loop
            # arms, was most of the pilot B to E gap, and the baseline had no
            # contractual way to earn it. On a null the correct behaviour is to
            # conclude benign, so asking is a wrong first action and scores as
            # one. Over-asking remains reported separately as over_asking and
            # ask_rate, where it cannot inflate a declared contrast.
            if first.get('action') == 'request':
                gate = 0.0
            elif gate == 1.0 and str(first.get('verdict','')).lower() != 'benign':
                gate = 0.0

    # D2 resolution efficiency. Reaching the right terminal state in fewer
    # tiers is better. Failing to ask when the case required it scores zero.
    if case['resolvable']:
        ok_terminal = rec['terminal_state'] == 'concluded'
        eff = max(0.0, 1.0 - 0.25*(rec['iters']-1)) if ok_terminal else 0.0
    else:
        eff = 1.0 if rec['terminal_state'] == 'escalated' else 0.0

    # collection targeting. Only meaningful where a resolving field exists.
    reqs = [(h.get('parsed') or {}).get('requested_fields') or []
            for h in rec['history']]
    req  = ' '.join(str(x) for x in sum(reqs, [])).lower()
    # CORRECTED 2026-08-04 from pilot 1. The previous version matched the first
    # underscore-separated word of resolving_field. U01 therefore scored a hit
    # on any request containing the word "process", which llama satisfied on
    # every attempt, while A01 could only score by containing the string "rmm",
    # so deepseek's correct request about the change record and the blocklist
    # entry scored zero. It was rewarding a vocabulary coincidence and
    # penalising correct requests phrased in analyst language.
    # CORRECTED AGAIN 2026-08-04 after pilot 2. A trial that made no request at
    # all was scoring 0.0, which conflates not asking with asking badly. On a
    # null case not asking is the correct behaviour, so the measure penalised
    # restraint and rewarded whichever model asked most often: llama led at
    # 0.500 largely by asking on everything, repeatedly. Asking rate is now a
    # separate measure and targeting is conditional on a request existing.
    asked = 1.0 if any(reqs) else 0.0
    if not case['resolving_field'] or not ARMS[rec['arm']]['loop'] or not asked:
        targeted = np.nan   # no key, arm cannot request, or nothing was asked
    else:
        # Word-set matching, not substring. Pilot 3: qwen asked for
        # 'script_file_hash' while the alias read 'script hash', which is not a
        # substring of it, so a correctly targeted request scored zero.
        # Underscores are normalised and an alias matches when all of its words
        # are present somewhere in the request text.
        bag = set(w for w in re.split(r'[^a-z0-9]+', req.replace('_', ' ')) if w)
        targeted = 0.0
        for a in (case.get('resolving_aliases') or []):
            words = [w for w in re.split(r'[^a-z0-9]+',
                                         a.lower().replace('_', ' ')) if w]
            if words and all(w in bag for w in words):
                targeted = 1.0
                break

    # Repeat requests. Pilot 1 showed llama reissuing an identical field list
    # after new telemetry had arrived, on N01, A01 and P01. A loop that asks
    # again for what it was just given is not collecting, it is stalling.
    norm = [tuple(sorted(str(x).lower().strip() for x in r)) for r in reqs if r]
    repetition = 1.0 if len(norm) > len(set(norm)) else 0.0

    # fabricated fields. Both comparisons are scored on every trial: the strict
    # one is what produced a 0.32 rate on cosmetics, and keeping it visible is
    # the only protection against a lenient comparison hiding a later change.
    disclosed = disclosed_text(case, rec['final_tier'])
    bad_quotes        = verify_quotes(f, disclosed) if f else []
    bad_quotes_strict = verify_quotes(f, disclosed, strict=True) if f else []

    # falsifier, three of four sub-measures are mechanical.
    # CORRECTED 2026-08-05 alongside the Cell 7 schema split. Scoring an arm
    # 0.0 for failing to produce a field its own contract never asked for
    # repeats the gate_correctness defect one measure down: it would measure
    # the manipulation rather than the reasoning, and would replace a leak
    # reading 1.000 on every arm with an artefact reading 0.000 on four of
    # them. Structural unavailability is NaN. Behavioural non-arrival, meaning
    # an arm that was asked and did not answer, is still 0.0.
    fal = (f.get('falsifier') or {}) if f else {}
    if 'falsifier' not in ARM_HARNESS_FIELDS[rec['arm']]:
        f_present = f_checkable = np.nan
    else:
        f_present   = 1.0 if fal.get('statement') else 0.0
        f_checkable = 1.0 if (fal.get('check') and fal.get('data_needed')) else 0.0
    # Unprompted falsifiers. A model that states one where nothing asked for it
    # is the only remaining way this measure can say anything about an arm that
    # carries no falsifier field, so it is counted rather than discarded. If it
    # is not near zero on A, B, C and G, the harness accounts for less of the
    # effect than the arm labels imply.
    f_unprompted = (1.0 if (fal.get('statement') and
                            'falsifier' not in ARM_HARNESS_FIELDS[rec['arm']])
                    else 0.0)
    # f_discriminating is scored by the judge against flip_checks in Cell 12

    return {'gate_correctness': gate, 'resolution_efficiency': eff,
            'collection_targeting': targeted,
            'asked': asked,
            'request_repetition': repetition,
            'off_contract': 1.0 if rec['terminal_state']=='off_contract' else 0.0,
            'fabricated_quotes': len(bad_quotes),
            'fabricated_quotes_strict': len(bad_quotes_strict),
            'quotes_total': len((f.get('evidence_quotes') or [])) if f else 0,
            'falsifier_present': f_present, 'falsifier_checkable': f_checkable,
            'falsifier_unprompted': f_unprompted,
            'parse_failure': 1.0 if rec['terminal_state']=='parse_failure' else 0.0,
            'degenerate': 1.0 if any(is_degenerate(h.get('raw') or '')
                                     for h in rec['history']) else 0.0,
            'truncated': 1.0 if any(h['usage'].get('finish_reason') == 'length'
                                    for h in rec['history']) else 0.0,
            'tokens': rec['tokens'], 'iters': rec['iters'],
            'terminal_state': rec['terminal_state']}

# Step 14 · Cell 12: judge scoring


In [ ]:
# DECIDED 2026-08-05. One sentence of basis per score. At 40 measured output
# tokens the judge returned four integers and nothing checkable, so cause_match
# fed co-primary 2 unaudited and the independent scorer could only be compared
# with it numerically, which makes kappa a comparison of numbers rather than of
# reasoning. Cost is roughly 4x judge output from a measured base of 40 tokens,
# so the full grid still costs less than Cell 8 printed before judge_in and
# judge_out were measured. Take the new judge_out from Cell 12b, not by guess.
JUDGE_SYSTEM = textwrap.dedent("""\
  You are scoring a security triage output against a reference key.
  Return JSON only.
    cause_match      0 to 2   does the stated cause match the reference cause
    defensible       0 to 2   is the conclusion supported by the evidence cited
    falsifier_discriminating  0 or 1, or null if the output contains no
        falsifier field at all. 1 only if the falsifier names a check that
        appears in, or is equivalent to, one of the reference flip_checks. A
        falsifier that could never fire given this data scores 0. Do not
        penalise an output that was never asked for a falsifier: return null.
    unsupported_claims  integer count of assertions not supported by the
        telemetry shown
  For each of the four scores, also return a basis field naming the specific
  words in the output and in the reference that decided it:
    cause_match_basis, defensible_basis,
    falsifier_discriminating_basis, unsupported_claims_basis
  One sentence each. A score with no stated basis cannot be audited and cannot
  be disputed by the independent scorer.
  """)

def judge_trial(rec, case, disclosed):
    f = rec.get('final') or {}
    payload = {
      'telemetry_shown': disclosed,
      'model_output': f,
      'reference': {'true_verdict': case['true_verdict'],
                    'true_cause'  : case['true_cause'],
                    'flip_checks' : case['flip_checks']},
    }
    raw, usage = run_one(JUDGE_MODEL, JUDGE_SYSTEM,
                         json.dumps(payload, indent=1),
                         max_tokens=500, temperature=0.0)
    obj, err = parse_output_loose(raw)
    return obj or {}, usage

def parse_output_loose(text):
    i, j = text.find('{'), text.rfind('}')
    if i < 0 or j <= i: return None, 'no json'
    try: return json.loads(text[i:j+1]), None
    except Exception as e: return None, str(e)

# Step 15 · Cell 10b: pilot readout


In [ ]:
# PILOT READOUT. Run after Cell 11 so score_mechanical is defined.
recs, trailing = read_ckpt(CKPT / 'gen.jsonl')
assert recs and not trailing, 'no clean checkpoint to read'
by_id = {c['case_id']: c for c in CASES}

calls = []
for r in recs:
    for h in r['history']:
        calls.append({'model': r['model'], 'arm': r['arm'],
                      'case_id': r['case_id'], 'iter': h['iter'],
                      'in_tok' : h['usage']['prompt_tokens'],
                      'out_tok': h['usage']['completion_tokens'],
                      'truncated': h['usage'].get('finish_reason') == 'length',
                      'parsed_ok': h['parsed'] is not None})
cdf = pd.DataFrame(calls)

print('MEASURED TOKENS, these replace the TOK placeholders in Cell 8')
print(f'  gen_in      {cdf.in_tok.mean():7.0f}   in Cell 8: {TOK["gen_in"]}')
print(f'  gen_out     {cdf.out_tok.mean():7.0f}   in Cell 8: {TOK["gen_out"]}')
print(f'  gen_out p95 {cdf.out_tok.quantile(0.95):7.0f}   vs cap {MAX_TOKENS_PER_CALL}')
print(f'  parse rate  {cdf.parsed_ok.mean():7.2f}   must be near 1.00 now')

tdf  = pd.DataFrame([{k: v for k, v in r.items() if k != 'history'} for r in recs])
loop = tdf[tdf.arm.isin(LOOP_ARMS)]
print(f'  AVG_ITERS on loop arms {loop.iters.mean():.2f}   in Cell 8: {AVG_ITERS}')

scored = pd.DataFrame([{'arm': r['arm'], 'model': r['model'],
                        'case_type': by_id[r['case_id']]['case_type'],
                        **score_mechanical(r, by_id[r['case_id']])}
                       for r in recs])

print(NL + 'PER ARM. Truncation and parse failure must not differ by arm. The\
AXIOM harness demands the longest output of any arm, so if it truncates and\
the baseline does not, the comparison measures headroom, not reasoning.')
display(scored.groupby('arm')[['parse_failure','truncated','iters','tokens',
                               'gate_correctness']].mean())

print(NL + 'PARSE FAILURE DIAGNOSTICS. Pilots 1 to 3 read an arm gradient into')
print('counts of 3, 5 and 6 failures out of 24, and the parser was changed')
print('twice on that reading. The cause was one generator emitting one word')
print('per line. Replacing it took the rate to 1.00 with no change to any arm.')
print('Read the error strings and raw text below before touching the parser,')
print('and treat single-digit counts as noise until shown otherwise.')
diag = pd.DataFrame([{'arm': r['arm'], 'model': r['model'],
                      'case_id': r['case_id'], 'iter': h['iter'],
                      'err'     : h.get('parse_error') or 'ok',
                      'strategy': h.get('parse_strategy') or 'none',
                      'finish'  : h['usage'].get('finish_reason'),
                      'out_tok' : h['usage']['completion_tokens'],
                      'raw'     : h.get('raw') or ''}
                     for r in recs for h in r['history']])
display(pd.crosstab(diag.arm, diag.strategy))
display(diag[diag.err != 'ok'].groupby(['arm', 'err']).size())
for _, row in diag[diag.strategy == 'none'].head(8).iterrows():
    print(NL + f'--- {row.arm} {row.model} {row.case_id} iter={row.iter} '
          f'finish={row.finish} out_tok={row.out_tok}')
    print(repr(row.raw)[:700])

print(NL + 'TERMINAL STATES. cap_reached on a resolvable case means the loop')
print('spent every iteration without arriving at an answer.')
display(pd.crosstab(scored.arm, scored.terminal_state))
display(scored.groupby('arm')[['asked','request_repetition','off_contract']].mean())

print(NL + 'DG1. collection_targeting, scored only where a request was made')
res = scored[scored.collection_targeting.notna()]
if res.empty or res.collection_targeting.nunique() <= 1:
    print('  NO VARIANCE. The measure is dead as specified. It must be '
          'redefined or removed from the pre-registration before the full '
          'grid runs. Do not discover this after generating.')
else:
    display(res.groupby(['arm','model']).collection_targeting.agg(['mean','count']))

print(NL + 'Every requested_fields list actually emitted')
any_req = False
for r in recs:
    for h in r['history']:
        rf = (h.get('parsed') or {}).get('requested_fields') or []
        if rf:
            any_req = True
            print(f"  {r['arm']}  {r['model']:9s} {r['case_id']}  {rf}")
if not any_req:
    print('  none. No model asked for anything under any arm, including E. '
          'The loop never engaged, so C, D, E and F collapse onto B and F.')

In [ ]:
# Judge token measurement. Run after Cell 12. The judge and the blinding
# auditor together are roughly four fifths of the full grid cost, so judge_in
# and judge_out are worth more than the generator numbers above.
import random as _r
_sample = _r.Random(SEED).sample(recs, min(6, len(recs)))
_ji, _jo = [], []
for r in _sample:
    case = by_id[r['case_id']]
    _obj, _u = judge_trial(r, case, disclosed_text(case, r['final_tier']))
    _ji.append(_u['prompt_tokens']); _jo.append(_u['completion_tokens'])
print(f'  judge_in  {np.mean(_ji):7.0f}   placeholder 1600')
print(f'  judge_out {np.mean(_jo):7.0f}   placeholder  300')
print('  audit_in and audit_out come from the Cell 13 blinding call, which is '
      'much shorter. Measure it the same way before the full run.')

# Step 16 · Cell 12b: build the judged table


In [ ]:
def judge_all(path=CKPT/'judge.jsonl'):
    """One judge call per trial, checkpointed on output_id so a disconnect does
    not repay for work already done."""
    by_id = {c['case_id']: c for c in CASES}
    done  = {}
    if path.exists():
        for line in path.open():
            r = json.loads(line); done[r['output_id']] = r
    recs, _ = read_ckpt(CKPT / 'gen.jsonl')
    rows = []
    with path.open('a') as fh:
        for rec in recs:
            oid = f"{rec['model']}_{rec['arm']}_{rec['case_id']}_{rec['rep']}"
            if oid in done:
                rows.append(done[oid]); continue
            case = by_id[rec['case_id']]
            obj, u = judge_trial(rec, case,
                                 disclosed_text(case, rec['final_tier']))
            row = {'output_id': oid, 'model': rec['model'], 'arm': rec['arm'],
                   'case_id': rec['case_id'], 'case_type': case['case_type'],
                   'cause_match': obj.get('cause_match'),
                   'defensible' : obj.get('defensible'),
                   'falsifier_discriminating': obj.get('falsifier_discriminating'),
                   'unsupported_claims': obj.get('unsupported_claims'),
                   # Added 2026-08-05. Kept on the checkpoint so the human
                   # scorer comparison can be read on reasoning rather than on
                   # four integers, and so any judge score can be disputed.
                   'cause_match_basis': obj.get('cause_match_basis'),
                   'defensible_basis' : obj.get('defensible_basis'),
                   'falsifier_discriminating_basis':
                       obj.get('falsifier_discriminating_basis'),
                   'unsupported_claims_basis': obj.get('unsupported_claims_basis'),
                   'judge_in' : u['prompt_tokens'],
                   'judge_out': u['completion_tokens']}
            fh.write(json.dumps(row, default=str) + NL); fh.flush()
            rows.append(row)
    j = pd.DataFrame(rows)
    # No missing values rule. A judge that returned nothing for a trial scores
    # zero on that trial, it does not drop out of the denominator.
    # No missing values rule, with the structural exception stated in Cell 9.
    # A judge that returned nothing for a trial scores zero and stays in the
    # denominator. falsifier_discriminating is exempt: on an arm whose contract
    # carries no falsifier field there is nothing to score, and zero there
    # would be the same artefact the Cell 11 correction of 2026-08-05 removes.
    for c in ['cause_match','defensible','unsupported_claims']:
        j[c] = pd.to_numeric(j[c], errors='coerce').fillna(0)
    j['falsifier_discriminating'] = pd.to_numeric(
        j['falsifier_discriminating'], errors='coerce')
    _asked_fal = j.arm.isin([a for a in ARMS
                             if 'falsifier' in ARM_HARNESS_FIELDS[a]])
    j.loc[_asked_fal, 'falsifier_discriminating'] = (
        j.loc[_asked_fal, 'falsifier_discriminating'].fillna(0))
    print(f'judge_in  {j.judge_in.mean():7.0f}   in Cell 8: {TOK["judge_in"]}')
    print(f'judge_out {j.judge_out.mean():7.0f}   in Cell 8: {TOK["judge_out"]}')
    return j

judged   = judge_all()
judge_df = judged.set_index('output_id').sort_index()

# The independent scorer's returned sheet, keyed on output_id. Until a scorer
# is recruited this file does not exist, and method rule 5 is unmet. Cell 18
# must say so and return NO VERDICT rather than raise.
try:
    human_df = (pd.read_csv(OUT / 'human_returned.csv')
                  .set_index('output_id').sort_index())
except FileNotFoundError:
    human_df = None
    print('no returned human sheet at', OUT / 'human_returned.csv',
          '- method rule 5 unmet, so no verdict is available. This is a fact '
          'about the study, not a bug.')

# Step 17 · Cell 13: blinding audit, fixed deidentify()


In [ ]:
AXIOM_TELLS = ['E0','E1','E2','evidence gate','competing explanation',
               'distinguishing evidence','falsifier','step 0','gate']

def deidentify(final_obj):
    """Render any arm's output into one uniform prose form.
    Structure, ordering and vocabulary must not identify the arm."""
    if not final_obj: return 'No parseable output was produced.'
    v = str(final_obj.get('verdict','')).lower()
    c = str(final_obj.get('cause','') or '')
    for t in AXIOM_TELLS:
        c = re.sub(re.escape(t), '', c, flags=re.I)
    c = ' '.join(c.split())
    conf = final_obj.get('confidence')
    conf_band = ('low' if isinstance(conf,(int,float)) and conf < 0.4 else
                 'medium' if isinstance(conf,(int,float)) and conf < 0.75 else
                 'high' if isinstance(conf,(int,float)) else 'unstated')
    n_q = len(final_obj.get('evidence_quotes') or [])
    return (f'The assessment is {v or "unstated"}. '
            f'The stated cause is: {c or "none given"}. '
            f'Confidence is {conf_band}. '
            f'{n_q} pieces of evidence were cited.')

def blinding_audit(df_blinded):
    """Auditor model guesses the arm. Gate fails if it can."""
    recalls = {}
    for arm in df_blinded.arm.unique():
        sub = df_blinded[df_blinded.arm==arm]
        hits = 0
        for _, r in sub.iterrows():
            g, _ = run_one(JUDGE_MODEL,
                 'Guess which of A B C D E F G produced this. One letter only.',
                 r.blinded, max_tokens=5, temperature=0.0)
            hits += int(g.strip().upper().startswith(arm))
        recalls[arm] = hits/len(sub)
    ok = max(recalls.values()) < GATES['blinding_max_recall']
    return ok, recalls

# Step 18 · Cell 14: human blind scoring sheet, with reference key


In [ ]:
def export_human_sheet(df, path=OUT/'human_sheet.csv'):
    """v0.3 exported blinded_output only, so human and judge scored against
    different information and kappa measured information asymmetry rather
    than disagreement. The reference key travels with the sheet."""
    cols = pd.DataFrame({
      'output_id'      : df.output_id,
      'telemetry_shown': df.disclosed,
      'blinded_output' : df.blinded,
      'ref_true_verdict': df.true_verdict,
      'ref_true_cause'  : df.true_cause,
      'ref_flip_checks' : df.flip_checks.apply(lambda x: ' | '.join(x)),
      'cause_match'     : '',   # 0-2 ordinal
      'defensible'      : '',   # 0-2 ordinal
      'falsifier_discriminating': '',  # 0/1 binary
      'unsupported_claims': '', # count
      'key_dispute'     : '',   # free text: flag a wrong reference key
    })
    cols.sample(frac=1, random_state=SEED).to_csv(path, index=False)
    return path

def kappas(human, judge):
    """Split by scale type. Quadratic for ordinal, unweighted for binary,
    Spearman for counts. Averaging across types was a v0.3 defect."""
    from sklearn.metrics import cohen_kappa_score
    from scipy.stats import spearmanr
    out = {}
    for c in ['cause_match','defensible']:
        out[c] = cohen_kappa_score(human[c], judge[c], weights='quadratic')
    out['falsifier_discriminating'] = cohen_kappa_score(
        human['falsifier_discriminating'], judge['falsifier_discriminating'])
    out['unsupported_claims'] = spearmanr(human['unsupported_claims'],
                                          judge['unsupported_claims']).statistic
    ordinal_mean = np.mean([out['cause_match'], out['defensible']])
    return out, ordinal_mean >= GATES['irr_min_kappa']

# Step 19 · Cell 15: trial table, no missing values


In [ ]:
def assemble(gen_path=CKPT/'gen.jsonl'):
    by_id = {c['case_id']: c for c in CASES}
    rows = []
    recs, trailing = read_ckpt(Path(gen_path))
    if trailing:
        raise SystemExit(f'{trailing} unparseable bytes at the end of {gen_path}. '
                         'Rerun generate_all to complete the missing trials before '
                         'scoring. Scoring a truncated file drops trials silently, '
                         'and that loss is not randomly distributed across arms.')
    for rec in recs:
        case = by_id[rec['case_id']]
        disclosed = disclosed_text(case, rec['final_tier'])
        mech = score_mechanical(rec, case)
        rows.append({
          'output_id': f"{rec['model']}_{rec['arm']}_{rec['case_id']}_{rec['rep']}",
          'model': rec['model'], 'arm': rec['arm'], 'case_id': rec['case_id'],
          'case_type': case['case_type'], 'rep': rec['rep'],
          'resolvable': case['resolvable'],
          'true_verdict': case['true_verdict'], 'true_cause': case['true_cause'],
          'flip_checks': case['flip_checks'],
          'disclosed': disclosed,
          'blinded': deidentify(rec.get('final')),
          'verdict': (rec.get('final') or {}).get('verdict'),
          'confidence': (rec.get('final') or {}).get('confidence'),
          **mech,
        })
    return pd.DataFrame(rows)

df = assemble()
# resolution_efficiency applies to every arm. gate_correctness is NaN only
# where the arm has no contractual way to take the expected action, which is a
# property of the design rather than a missing value. Any other NaN is a defect
# and must stop the analysis.
assert df.resolution_efficiency.notna().all(), 'missing resolution_efficiency'
_g = df[df.gate_correctness.isna()]
assert (~_g.arm.map(lambda a: ARMS[a]['loop'])).all(), \
    'gate_correctness missing on an arm that could have acted'
print(df.groupby(['arm','case_type']).size().unstack(fill_value=0))

case_type  adversarial_null  clean_null  true_positive  underdetermined
arm                                                                    
A                         6           6              6                6
B                         6           6              6                6
E                         6           6              6                6


# Step 20 · Cell 16: co-primaries and miss-by-silence


In [ ]:
NULLS = ['clean_null','adversarial_null']

def fp_rate_on_nulls(d):
    s = d[d.case_type.isin(NULLS)]
    conf = s.confidence.fillna(0.0)
    return float(((s.verdict=='malicious') & (conf >= TAU)).mean())

def correct_cause_on_tp(d, judged):
    s = judged[judged.case_type=='true_positive']
    return float((s.cause_match >= 2).mean())

def miss_by_silence(d):
    """Abstention on a true positive. This is the framework's own
    false-closure. Report it whether or not it is flattering."""
    s = d[d.case_type=='true_positive']
    return float(s.terminal_state.isin(['escalated','cap_reached']).mean())

def over_asking(d):
    s = d[d.case_type.isin(NULLS)]
    return float((s.iters > 1).mean())

summary = df.groupby('arm').apply(lambda d: pd.Series({
    'fp_rate_nulls'   : fp_rate_on_nulls(d),
    'miss_by_silence' : miss_by_silence(d),
    'over_asking'     : over_asking(d),
    'mean_iters'      : d.iters.mean(),
    'mean_tokens'     : d.tokens.mean(),
    'fabricated_rate' : d.fabricated_quotes.sum()/max(1,d.quotes_total.sum()),
    # The same rate under the strict comparison. In the pilot 5 rescore these
    # read 0.00 and 0.32 respectively, and the gap is entirely backslash count
    # and letter case. If they ever diverge for another reason, that is a real
    # finding about the generator and must not be invisible.
    'fabricated_strict': d.fabricated_quotes_strict.sum()/max(1,d.quotes_total.sum()),
    'truncated'       : d.truncated.mean(),
    'parse_failure'   : d.parse_failure.mean(),
    'degenerate'      : d.degenerate.mean(),
    'off_contract'    : d.off_contract.mean(),
    'ask_rate'        : d.asked.mean(),
    'repeat_requests' : d.request_repetition.mean(),
    'gate_correct'    : d.gate_correctness.mean(),
    'resolution_eff'  : d.resolution_efficiency.mean(),
    'targeting'       : d.collection_targeting.mean(),
    'falsifier_present': d.falsifier_present.mean(),
    # Added 2026-08-04 after pilot 5. falsifier_present read 1.000 on every arm
    # including the bare baseline. That is not a harness effect: the shared
    # TRIAGE_SCHEMA_DOC asks every arm for a falsifier object, so step 3 of the
    # AXIOM harness is handed to arm A by the output contract. Presence cannot
    # discriminate and should not be reported as though it could. Checkability
    # is the weakest measure that still can, and discrimination against
    # flip_checks is the judge's job in Cell 12.
    'falsifier_checkable': d.falsifier_checkable.mean(),
    # Added 2026-08-05. On arms carrying no falsifier field the two measures
    # above are NaN by design, so this column is the only place an unrequested
    # falsifier from a bare arm can appear.
    'falsifier_unprompted': d.falsifier_unprompted.mean(),
    # Denominator for fabricated_rate. A rate of 0.32 on six quotes means
    # something different from 0.32 on sixty.
    'quotes_total'     : d.quotes_total.sum(),
    'quotes_bad'       : d.fabricated_quotes.sum(),
}), include_groups=False)

# Co-primary 2, added 2026-08-04. The table above carried nineteen columns and
# not one of them was correct_cause_on_tp. Both co-primaries are declared to be
# reported always and never averaged, and putting one in the headline table
# while leaving the other to the contrast cell is exactly how a specificity gain
# paid for with sensitivity reaches print. It needs the judged table, so it is
# joined here rather than computed inside the groupby.
if 'judged' in globals():
    summary.insert(1, 'correct_cause_tp', pd.Series(
        {a: correct_cause_on_tp(df[df.arm == a], judged[judged.arm == a])
         for a in summary.index}))
else:
    print('judged table absent, so co-primary 2 is missing from this summary. '
          'Run Cell 12b. Do not read the specificity column on its own.')
summary

# Step 21 · Cell 16b: quote integrity and null diagnostics


In [ ]:
recs, _ = read_ckpt(CKPT / 'gen.jsonl')
by_id   = {c['case_id']: c for c in CASES}

pairs = []
for rec in recs:
    case = by_id[rec['case_id']]
    disc = disclosed_text(case, rec['final_tier'])
    for q in ((rec.get('final') or {}).get('evidence_quotes') or []):
        if _qnorm(q, strict=True) in _qnorm(disc, strict=True):
            continue
        qbs, dbs = ' '.join(str(q).split()), ' '.join(disc.split())
        while BS + BS in qbs: qbs = qbs.replace(BS + BS, BS)
        while BS + BS in dbs: dbs = dbs.replace(BS + BS, BS)
        pairs.append({'arm': rec['arm'], 'model': rec['model'],
                      'case_id': rec['case_id'],
                      'passes_if_backslashes_collapsed': qbs in dbs,
                      'passes_if_case_folded':
                          ' '.join(str(q).split()).casefold()
                          in ' '.join(disc.split()).casefold(),
                      'passes_lenient': _qnorm(q) in _qnorm(disc),
                      'quote': ' '.join(str(q).split())[:90]})
qdf = pd.DataFrame(pairs)
print(f'{len(qdf)} failing quotes of {int(df.quotes_total.sum())} cited')
if len(qdf):
    display(qdf.groupby(['arm','passes_if_backslashes_collapsed']).size())
    display(qdf.head(20))

print(NL + 'FALSE FINDINGS ON NULLS by case and by model. Which null case does')
print('the work decides whether A01 is adversarial after all, and whether the')
print('pilot 4 claim that this corpus is too easy for the baseline was wrong.')
_n = df[df.case_type.isin(NULLS)].copy()
_n['false_finding'] = ((_n.verdict == 'malicious') &
                       (_n.confidence.fillna(0.0) >= TAU)).astype(float)
display(_n.pivot_table(index='arm', columns='case_id', values='false_finding'))
display(_n.pivot_table(index='model', columns='arm', values='false_finding'))

# Step 22 · Cell 17: clustered bootstrap on the declared contrasts


In [ ]:
CONTRASTS = [('B','A'),('C','B'),('E','C'),('E','D'),('E','F'),('E','G')]

def boot_contrast(df, judged, fn, a1, a0, n=BOOTSTRAP_N):
    """Resample CASES with replacement, not observations. Observations cluster
    within case, so resampling rows is anticonservative. This was already a
    stated weakness of the v0.3 Fisher test.

    CORRECTED 2026-08-04 after pilot 5, where every contrast including B-A
    returned nan. Two silent causes:
      1. Uniform resampling of case ids can draw a set containing no null case
         at all. On a four case pilot that happens on roughly 6 percent of
         draws, fp_rate_on_nulls is nan on those draws, and a single nan in
         5000 makes the mean and both percentiles nan. Resampling is now
         stratified within case_type, which is also the correct choice on this
         design: the case mix is fixed by the pre-registration, not sampled.
      2. A contrast naming an arm absent from the run produced an empty frame
         and therefore a number-shaped nan. Absent arms now say so.
    """
    have = set(df.arm.unique())
    if a1 not in have or a0 not in have:
        return {'status': 'arm absent from this run: '
                          f'{sorted({a1, a0} - have)}'}
    strata = df.groupby('case_type').case_id.unique().to_dict()
    # CORRECTED AGAIN 2026-08-04 after the pilot 5 rescore. Stratification
    # removed the nan and replaced it with something more dangerous. With one
    # case per type every draw returns that same case, so B-A came back as
    # diff -0.333 with lo and hi both -0.333 and excludes_zero True. A zero
    # width interval that excludes zero reads exactly like a decisive result.
    # nan announces itself; this does not. No interval is returned until every
    # case type carries at least two cases.
    thin = sorted(t for t, v in strata.items() if len(v) < 2)
    if thin:
        return {'status': 'degenerate resample, no interval: only one case of '
                          f'type {thin}. Every bootstrap draw is identical, so '
                          'the interval has zero width and excludes_zero is '
                          'meaningless rather than significant.'}
    diffs  = np.full(n, np.nan)
    for i in range(n):
        pick = np.concatenate([np.random.choice(v, size=len(v), replace=True)
                               for v in strata.values()])
        sub  = pd.concat([df[df.case_id == c] for c in pick])
        jsub = (pd.concat([judged[judged.case_id == c] for c in pick])
                if judged is not None else None)
        if jsub is not None:
            diffs[i] = fn(sub[sub.arm == a1], jsub) - fn(sub[sub.arm == a0], jsub)
        else:
            diffs[i] = fn(sub[sub.arm == a1]) - fn(sub[sub.arm == a0])
    n_bad = int(np.isnan(diffs).sum())
    good  = diffs[~np.isnan(diffs)]
    if len(good) < 0.9 * n:
        return {'status': f'{n_bad} of {n} resamples undefined. Too few cases '
                          'of the relevant type for a stable interval.'}
    lo, hi = np.percentile(good, [2.5, 97.5])
    return {'diff': float(good.mean()), 'lo': float(lo), 'hi': float(hi),
            'excludes_zero': bool(lo > 0 or hi < 0),
            'undefined_resamples': n_bad}

results = {}
for a1, a0 in CONTRASTS:
    r = {'fp_nulls': boot_contrast(df, None, fp_rate_on_nulls, a1, a0)}
    # Co-primary 2 is reported whenever the judged table exists. Reporting one
    # co-primary and not the other is how a specificity gain paid for with
    # sensitivity gets published.
    if 'judged' in globals():
        r['correct_cause'] = boot_contrast(df, judged, correct_cause_on_tp,
                                           a1, a0)
    results[f'{a1}-{a0}'] = r
pd.DataFrame(results).T

# Step 23 · Cell 18: gates, verdict, artefact


In [ ]:
def verdict():
    blind_ok, recalls = blinding_audit(df)
    # CORRECTED 2026-08-04 after pilot 5, where this line raised NameError
    # because nothing built human_df or judge_df. The absence of an independent
    # scorer is a disclosed hole in this study, not an exception: it must
    # produce NO VERDICT with a stated reason. A crash here invites someone to
    # comment the gate out and report a result anyway.
    if human_df is None or judge_df is None:
        kap, irr_ok = {'note': 'no independent scorer, kappa not computable'}, False
    else:
        common = human_df.index.intersection(judge_df.index)
        if len(common) < len(judge_df):
            print(f'WARNING: human sheet covers {len(common)} of '
                  f'{len(judge_df)} judged outputs')
        kap, irr_ok = kappas(human_df.loc[common], judge_df.loc[common])
    fab_ok = (df.fabricated_quotes.sum()/max(1,df.quotes_total.sum())) < GATES['fabricated_field_max']

    validity = {'blinding_ok': blind_ok, 'irr_ok': irr_ok, 'quote_integrity_ok': fab_ok}
    if not all(validity.values()):
        return {'verdict':'NO VERDICT', 'validity':validity, 'recalls':recalls,
                'kappas':kap,
                'note':'validity gates failed. No outcome claim may be made. '
                       'This is a result about the instrument, not the framework.'}

    E_fp = fp_rate_on_nulls(df[df.arm=='E']); A_fp = fp_rate_on_nulls(df[df.arm=='A'])
    E_cc = correct_cause_on_tp(df[df.arm=='E'], judged[judged.arm=='E'])
    A_cc = correct_cause_on_tp(df[df.arm=='A'], judged[judged.arm=='A'])

    not_worse = (E_fp <= A_fp + 0.05) and (E_cc >= 0.90 * A_cc)
    beats_B   = any(results[k]['fp_nulls']['excludes_zero'] for k in ['E-C'] ) \
                or results.get('C-B',{}).get('fp_nulls',{}).get('excludes_zero', False)

    if not not_worse:
        v = 'D. FRAMEWORK NET HARMFUL. Study falsifier 3 fired. Stop.'
    elif not beats_B:
        v = 'C. PERMISSION ONLY. Study falsifier 1 fired. The working component is the abstain sentence, not the harness.'
    elif results['E-G']['fp_nulls']['diff'] <= 0:
        v = 'B. LOOP IS AN INTEGRATION WORKAROUND. Study falsifier 2 fired. Full disclosure matches or beats the loop.'
    else:
        v = 'A. LOOP AND HARNESS BOTH CONTRIBUTE. Promote to SL v0.2 and seek held-out replication.'

    return {'verdict': v, 'validity': validity, 'E_fp': E_fp, 'A_fp': A_fp,
            'E_cc': E_cc, 'A_cc': A_cc, 'contrasts': results,
            'miss_by_silence': miss_by_silence(df[df.arm=='E'])}

out = {'design_hash': DESIGN_HASH, 'witness': EXTERNAL_WITNESS,
       'summary': summary.to_dict(), 'verdict': verdict()}
(OUT/'result.json').write_text(json.dumps(out, indent=2, default=str))
print(json.dumps(out['verdict'], indent=2, default=str))

{
  "verdict": "NO VERDICT",
  "validity": {
    "blinding_ok": false,
    "irr_ok": false,
    "quote_integrity_ok": "False"
  },
  "recalls": {
    "A": 0.16666666666666666,
    "B": 0.0,
    "E": 0.5416666666666666
  },
  "kappas": {
    "note": "no independent scorer, kappa not computable"
  },
  "note": "validity gates failed. No outcome claim may be made. This is a result about the instrument, not the framework."
}
